# CRIMEX Dataset Construction Notebook

## 1. Overview

This notebook starts the CRIMEX dataset pipeline using real data sources.

The objective is to:
- collect real crime-related data
- inspect and standardize the data
- identify enrichment opportunities
- build a reproducible foundation for the CRIMEX dataset

This notebook is designed to be:
- clean
- reproducible
- step-by-step
- research-friendly

In [1]:
# [1.1] Import libraries

import pandas as pd

C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 2. Load real crime dataset (fully reproducible)

We load a real crime dataset directly from a public source.

This ensures:
- no manual steps
- full reproducibility
- real-world data from the start

In [3]:
# [1.1] Setup

import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Allow importing local src modules
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Final folder:", FINAL_DIR)

Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks
Raw folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks\data\raw
Final folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks\data\final


In [4]:
# [2.1] Download full LA crime dataset with pagination

import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

base_url = "https://data.lacity.org/resource/2nrs-mtv8.csv"

limit = 50000
offset = 0
chunks = []

while True:
    url = f"{base_url}?$limit={limit}&$offset={offset}"
    chunk = pd.read_csv(url)

    print(f"Downloaded offset {offset}: {len(chunk)} rows")

    if chunk.empty:
        break

    chunks.append(chunk)

    if len(chunk) < limit:
        break

    offset += limit

df = pd.concat(chunks, ignore_index=True)

raw_path = RAW_DIR / "la_crime_full_raw_2.csv"
df.to_csv(raw_path, index=False)

print("Final shape:", df.shape)
print("Saved:", raw_path)

df.head()

Downloaded offset 0: 50000 rows
Downloaded offset 50000: 50000 rows
Downloaded offset 100000: 50000 rows
Downloaded offset 150000: 50000 rows
Downloaded offset 200000: 50000 rows
Downloaded offset 250000: 50000 rows
Downloaded offset 300000: 50000 rows
Downloaded offset 350000: 50000 rows
Downloaded offset 400000: 50000 rows
Downloaded offset 450000: 50000 rows
Downloaded offset 500000: 50000 rows
Downloaded offset 550000: 50000 rows
Downloaded offset 600000: 50000 rows
Downloaded offset 650000: 50000 rows
Downloaded offset 700000: 50000 rows
Downloaded offset 750000: 50000 rows
Downloaded offset 800000: 50000 rows
Downloaded offset 850000: 50000 rows
Downloaded offset 900000: 50000 rows
Downloaded offset 950000: 50000 rows
Downloaded offset 1000000: 4894 rows
Final shape: (1004894, 28)
Saved: data\raw\la_crime_full_raw_2.csv


,dr_no,date_rptd,date_occ,time_occ,area,area_name,rpt_dist_no,part_1_2,crm_cd,crm_cd_desc,...,status,status_desc,crm_cd_1,crm_cd_2,crm_cd_3,crm_cd_4,location,cross_street,lat,lon
0,211507896,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


## 3.1 Inspect dataset columns

We list all columns to understand the structure of the dataset.

In [5]:
# [3.1] Show columns

print("Number of columns:", len(df.columns))
print("\nColumns:\n")

for col in df.columns:
    print(col)

Number of columns: 28

Columns:

dr_no
date_rptd
date_occ
time_occ
area
area_name
rpt_dist_no
part_1_2
crm_cd
crm_cd_desc
mocodes
vict_age
vict_sex
vict_descent
premis_cd
premis_desc
weapon_used_cd
weapon_desc
status
status_desc
crm_cd_1
crm_cd_2
crm_cd_3
crm_cd_4
location
cross_street
lat
lon


## 3.2 Standardize column names

We rename columns to clear, consistent, and CRIMEX-friendly names.

This improves:
- readability
- consistency across datasets
- future merging and enrichment

In [6]:
# [3.2] Rename columns to standardized format

df = df.rename(columns={
    "dr_no": "case_id",
    "date_rptd": "date_reported",
    "date_occ": "date_occured",
    "time_occ": "time_occured",
    "area": "area_code",
    "area_name": "area_name",
    "rpt_dist_no": "report_district",
    "part_1_2": "crime_part",
    "crm_cd": "crime_code",
    "crm_cd_desc": "crime_description",
    "mocodes": "modus_operandi",
    "vict_age": "victim_age",
    "vict_sex": "victim_sex",
    "vict_descent": "victim_ethnicity",
    "premis_cd": "premise_code",
    "premis_desc": "premise_description",
    "weapon_used_cd": "weapon_code",
    "weapon_desc": "weapon_description",
    "status": "case_status",
    "status_desc": "case_status_description",
    "crm_cd_1": "crime_code_1",
    "crm_cd_2": "crime_code_2",
    "crm_cd_3": "crime_code_3",
    "crm_cd_4": "crime_code_4",
    "location": "street_address",
    "cross_street": "cross_street",
    "lat": "latitude",
    "lon": "longitude"
})

df.head()

,case_id,date_reported,date_occured,time_occured,area_code,area_name,report_district,crime_part,crime_code,crime_description,...,case_status,case_status_description,crime_code_1,crime_code_2,crime_code_3,crime_code_4,street_address,cross_street,latitude,longitude
0,211507896,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


## 3.3 Standardize date and time

We define a reusable function to convert date and time columns into proper formats.

In [7]:
# [3.3] Define function for date and time conversion

def standardize_datetime_columns(
    df,
    date_reported_col="date_reported",
    date_occured_col="date_occured",
    time_occured_col="time_occured"
):
    df = df.copy()

    # Convert date columns
    df[date_reported_col] = pd.to_datetime(df[date_reported_col], errors="coerce")
    df[date_occured_col] = pd.to_datetime(df[date_occured_col], errors="coerce")

    # Convert time column (HHMM → time)
    df[time_occured_col] = df[time_occured_col].astype(str).str.zfill(4)
    df[time_occured_col] = pd.to_datetime(
        df[time_occured_col],
        format="%H%M",
        errors="coerce"
    ).dt.time

    return df

## 3.4 Apply date and time standardization

We apply the datetime standardization function to the dataset.

In [8]:
# [3.4] Apply datetime standardization

df = standardize_datetime_columns(df)

df[["date_reported", "date_occured", "time_occured"]].head()

,date_reported,date_occured,time_occured
0,2021-04-11,2020-11-07,08:45:00
1,2020-10-21,2020-10-18,18:45:00
2,2024-12-10,2020-10-30,12:40:00
3,2020-12-24,2020-12-24,13:10:00
4,2020-10-03,2020-09-29,18:30:00


## 3.5 Create full datetime and temporal features

We combine date and time into one datetime column and extract core temporal features.

In [9]:
# [3.5] Define temporal feature function

def add_temporal_features(df):
    df = df.copy()

    df["datetime_occured"] = pd.to_datetime(
        df["date_occured"].astype(str) + " " + df["time_occured"].astype(str),
        errors="coerce"
    )

    df["year"] = df["datetime_occured"].dt.year
    df["quarter"] = df["datetime_occured"].dt.quarter
    df["month"] = df["datetime_occured"].dt.month
    df["month_name"] = df["datetime_occured"].dt.month_name()
    df["week_of_year"] = df["datetime_occured"].dt.isocalendar().week.astype("Int64")
    df["day"] = df["datetime_occured"].dt.day
    df["day_of_year"] = df["datetime_occured"].dt.dayofyear
    df["day_of_week_number"] = df["datetime_occured"].dt.dayofweek
    df["day_of_week_name"] = df["datetime_occured"].dt.day_name()
    df["hour"] = df["datetime_occured"].dt.hour
    df["is_weekend"] = df["day_of_week_number"].isin([5, 6])

    def get_season(month):
        if pd.isna(month):
            return None
        if month in [12, 1, 2]:
            return "winter"
        if month in [3, 4, 5]:
            return "spring"
        if month in [6, 7, 8]:
            return "summer"
        return "autumn"

    df["season"] = df["month"].apply(get_season)

    return df

## 3.6 Apply temporal features

We apply the temporal feature function to the dataset.

In [10]:
# [3.6] Apply temporal feature function

df = add_temporal_features(df)

df[[
    "datetime_occured",
    "year",
    "quarter",
    "month",
    "month_name",
    "week_of_year",
    "day",
    "day_of_year",
    "day_of_week_number",
    "day_of_week_name",
    "hour",
    "is_weekend",
    "season"
]].head()

,datetime_occured,year,quarter,month,month_name,week_of_year,day,day_of_year,day_of_week_number,day_of_week_name,hour,is_weekend,season
0,2020-11-07 08:45:00,2020,4,11,November,45,7,312,5,Saturday,8,True,autumn
1,2020-10-18 18:45:00,2020,4,10,October,42,18,292,6,Sunday,18,True,autumn
2,2020-10-30 12:40:00,2020,4,10,October,44,30,304,4,Friday,12,False,autumn
3,2020-12-24 13:10:00,2020,4,12,December,52,24,359,3,Thursday,13,False,winter
4,2020-09-29 18:30:00,2020,3,9,September,40,29,273,1,Tuesday,18,False,autumn


## 3.7 Check missing values

We inspect missing values to understand data quality and prepare for cleaning.

In [11]:
# [3.7] Check missing values

missing = df.isnull().sum().sort_values(ascending=False)

missing.head(15)

crime_code_4           1004830
crime_code_3           1002580
crime_code_2            935740
cross_street            850666
weapon_description      677678
weapon_code             677678
modus_operandi          151598
victim_ethnicity        144643
victim_sex              144631
premise_description        588
premise_code                16
crime_code_1                11
case_status                  1
quarter                      0
datetime_occured             0
dtype: int64

In [12]:
# [3.8] Drop highly missing columns

cols_to_drop = [
    "crime_code_4",
    "crime_code_3",
    "crime_code_2"
]

df = df.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("New shape:", df.shape)

Dropped columns: ['crime_code_4', 'crime_code_3', 'crime_code_2']
New shape: (1004894, 38)


## 3.9 Clean categorical fields

We clean and standardize categorical columns such as victim sex and ethnicity.

In [13]:
# [3.9] Clean categorical values

# Standardize victim_sex
df["victim_sex"] = df["victim_sex"].replace({
    "M": "male",
    "F": "female"
})

# Fill missing with 'unknown'
df["victim_sex"] = df["victim_sex"].fillna("unknown")

# Standardize victim_ethnicity (keep as is but fill missing)
df["victim_ethnicity"] = df["victim_ethnicity"].fillna("unknown")

# Quick check
df[["victim_sex", "victim_ethnicity"]].head(10)

,victim_sex,victim_ethnicity
0,male,H
1,male,H
2,male,W
3,female,A
4,male,H
5,male,B
6,female,B
7,female,H
8,male,W
9,male,W


## 3.10 Standardize victim ethnicity

We convert ethnicity codes into full descriptive labels.

In [14]:
# [3.10] Map ethnicity codes to readable values

ethnicity_map = {
    "H": "hispanic",
    "W": "white",
    "B": "black",
    "A": "asian",
    "C": "chinese",
    "D": "cambodian",
    "F": "filipino",
    "G": "guamanian",
    "I": "native_american",
    "J": "japanese",
    "K": "korean",
    "L": "laotian",
    "O": "other",
    "P": "pacific_islander",
    "S": "samoan",
    "U": "hawaiian",
    "V": "vietnamese",
    "X": "unknown"
}

df["victim_ethnicity"] = df["victim_ethnicity"].map(ethnicity_map)

# fill any remaining nulls
df["victim_ethnicity"] = df["victim_ethnicity"].fillna("unknown")

df[["victim_ethnicity"]].head(10)

,victim_ethnicity
0,hispanic
1,hispanic
2,white
3,asian
4,hispanic
5,black
6,black
7,hispanic
8,white
9,white


## 3.11 Create age group

We convert victim age into grouped categories and structured range codes.

In [15]:
# [3.11] Create age group with code and description

def get_age_group(age):
    if pd.isna(age) or age < 0:
        return "unknown", "unknown"

    if age <= 12:
        return "0_12", "child"
    elif age <= 19:
        return "13_19", "teen"
    elif age <= 35:
        return "20_35", "young_adult"
    elif age <= 60:
        return "36_60", "adult"
    else:
        return "60_plus", "senior"


df[["victim_age_group_code", "victim_age_group_desc"]] = df["victim_age"].apply(
    lambda x: pd.Series(get_age_group(x))
)

df[[
    "victim_age",
    "victim_age_group_code",
    "victim_age_group_desc"
]].head(10)

,victim_age,victim_age_group_code,victim_age_group_desc
0,31,20_35,young_adult
1,32,20_35,young_adult
2,30,20_35,young_adult
3,47,36_60,adult
4,63,60_plus,senior
5,35,20_35,young_adult
6,21,20_35,young_adult
7,14,13_19,teen
8,43,36_60,adult
9,57,36_60,adult


## 4.1 Check coordinate quality

We inspect latitude and longitude before starting geo enrichment.

In [16]:
# [4.1] Check latitude and longitude quality

print("Latitude nulls :", df["latitude"].isna().sum())
print("Longitude nulls:", df["longitude"].isna().sum())

print("\nLatitude range :", df["latitude"].min(), "to", df["latitude"].max())
print("Longitude range:", df["longitude"].min(), "to", df["longitude"].max())

df[["latitude", "longitude"]].head(10)

Latitude nulls : 0
Longitude nulls: 0

Latitude range : 0.0 to 34.3343
Longitude range: -118.6676 to 0.0


,latitude,longitude
0,34.2124,-118.4092
1,34.1993,-118.4203
2,34.1847,-118.4509
3,34.0339,-118.3747
4,33.9813,-118.4350
5,34.0830,-118.1678
6,34.0100,-118.2900
7,34.1107,-118.2589
8,34.2763,-118.5210
9,34.1493,-118.5886


## 4.2 Validate coordinates

We validate coordinates without dropping rows and create geo quality fields.

In [17]:
# [4.2] Define function to validate coordinates without dropping rows

def validate_coordinates(
    df,
    latitude_col="latitude",
    longitude_col="longitude"
):
    df = df.copy()

    df["latitude_original"] = df[latitude_col]
    df["longitude_original"] = df[longitude_col]

    valid_mask = (
        df[latitude_col].notna() &
        df[longitude_col].notna() &
        (df[latitude_col] != 0) &
        (df[longitude_col] != 0) &
        df[latitude_col].between(-90, 90) &
        df[longitude_col].between(-180, 180)
    )

    df["is_valid_coordinate"] = valid_mask
    df["latitude_final"] = df[latitude_col].where(valid_mask)
    df["longitude_final"] = df[longitude_col].where(valid_mask)

    df["geo_source"] = df["is_valid_coordinate"].map({
        True: "original_coordinate",
        False: "missing_or_invalid"
    })

    df["geo_precision_desc"] = df["is_valid_coordinate"].map({
        True: "exact_coordinate",
        False: "not_available"
    })

    return df

## 4.3 Apply coordinate validation

We apply coordinate validation and inspect the geo quality fields.

In [18]:
# [4.3] Apply coordinate validation

df = validate_coordinates(df)

print(df["is_valid_coordinate"].value_counts(dropna=False))

df[[
    "latitude_original",
    "longitude_original",
    "is_valid_coordinate",
    "latitude_final",
    "longitude_final",
    "geo_source",
    "geo_precision_desc"
]].head(10)

is_valid_coordinate
True     1002654
False       2240
Name: count, dtype: int64


,latitude_original,longitude_original,is_valid_coordinate,latitude_final,longitude_final,geo_source,geo_precision_desc
0,34.2124,-118.4092,True,34.2124,-118.4092,original_coordinate,exact_coordinate
1,34.1993,-118.4203,True,34.1993,-118.4203,original_coordinate,exact_coordinate
2,34.1847,-118.4509,True,34.1847,-118.4509,original_coordinate,exact_coordinate
3,34.0339,-118.3747,True,34.0339,-118.3747,original_coordinate,exact_coordinate
4,33.9813,-118.4350,True,33.9813,-118.4350,original_coordinate,exact_coordinate
5,34.0830,-118.1678,True,34.0830,-118.1678,original_coordinate,exact_coordinate
6,34.0100,-118.2900,True,34.0100,-118.2900,original_coordinate,exact_coordinate
7,34.1107,-118.2589,True,34.1107,-118.2589,original_coordinate,exact_coordinate
8,34.2763,-118.5210,True,34.2763,-118.5210,original_coordinate,exact_coordinate
9,34.1493,-118.5886,True,34.1493,-118.5886,original_coordinate,exact_coordinate


## 4.4 Reverse geocoding (test on small sample)

We use OpenStreetMap to extract city and country from coordinates.

In [19]:
# [4.4] Test reverse geocoding on small sample

import requests
import time

def reverse_geocode(lat, lon):
    url = "https://nominatim.openstreetmap.org/reverse"
    
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json"
    }

    headers = {
        "User-Agent": "crimex-project"
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        data = response.json()

        address = data.get("address", {})

        return {
            "city": address.get("city") or address.get("town") or address.get("village"),
            "country": address.get("country"),
            "country_code": address.get("country_code")
        }

    except:
        return {
            "city": None,
            "country": None,
            "country_code": None
        }


# Take only valid coordinates sample
sample_df = df[df["is_valid_coordinate"]].head(10).copy()

results = []

for _, row in sample_df.iterrows():
    geo = reverse_geocode(row["latitude_final"], row["longitude_final"])
    results.append(geo)
    time.sleep(1)  # important (avoid API block)

geo_df = pd.DataFrame(results)

sample_df = pd.concat([sample_df.reset_index(drop=True), geo_df], axis=1)

sample_df[[
    "latitude_final",
    "longitude_final",
    "city",
    "country",
    "country_code"
]]

,latitude_final,longitude_final,city,country,country_code
0,34.2124,-118.4092,Los Angeles,United States,us
1,34.1993,-118.4203,Los Angeles,United States,us
2,34.1847,-118.4509,Los Angeles,United States,us
3,34.0339,-118.3747,Los Angeles,United States,us
4,33.9813,-118.4350,Los Angeles,United States,us
5,34.0830,-118.1678,Los Angeles,United States,us
6,34.0100,-118.2900,Los Angeles,United States,us
7,34.1107,-118.2589,Los Angeles,United States,us
8,34.2763,-118.5210,Los Angeles,United States,us
9,34.1493,-118.5886,Los Angeles,United States,us


## 4.5 Define geo backend configuration

We define the geo backend settings so the pipeline can switch between online and local services.

In [20]:
# [4.5] Define geo backend configuration

GEO_CONFIG = {
    "geo_backend_code": "online",   # "online" or "local"
    "reverse_online_url": "https://nominatim.openstreetmap.org/reverse",
    "overpass_online_url": "https://overpass-api.de/api/interpreter",
    "reverse_local_url": "http://localhost:8080/reverse",
    "overpass_local_url": "http://localhost:12345/api/interpreter",
    "user_agent": "crimex-project",
    "request_timeout_seconds": 30,
    "sleep_seconds_online": 1.0,
    "sleep_seconds_local": 0.1
}

print(GEO_CONFIG)

{'geo_backend_code': 'online', 'reverse_online_url': 'https://nominatim.openstreetmap.org/reverse', 'overpass_online_url': 'https://overpass-api.de/api/interpreter', 'reverse_local_url': 'http://localhost:8080/reverse', 'overpass_local_url': 'http://localhost:12345/api/interpreter', 'user_agent': 'crimex-project', 'request_timeout_seconds': 30, 'sleep_seconds_online': 1.0, 'sleep_seconds_local': 0.1}


## 4.6 Select geo URLs by backend

We define a helper function to choose the correct reverse geocoding and POI URLs based on the selected backend.

In [21]:
# [4.6] Define function to select geo service URLs

def get_geo_service_urls(geo_config):
    backend = geo_config["geo_backend_code"]

    if backend == "online":
        return {
            "reverse_url": geo_config["reverse_online_url"],
            "overpass_url": geo_config["overpass_online_url"],
            "sleep_seconds": geo_config["sleep_seconds_online"]
        }

    if backend == "local":
        return {
            "reverse_url": geo_config["reverse_local_url"],
            "overpass_url": geo_config["overpass_local_url"],
            "sleep_seconds": geo_config["sleep_seconds_local"]
        }

    raise ValueError(f"Unsupported geo backend: {backend}")

## 4.7 Check selected geo service URLs

We verify which URLs and timing settings are active for the selected geo backend.

In [22]:
# [4.7] Check active geo service settings

geo_service_urls = get_geo_service_urls(GEO_CONFIG)

print(geo_service_urls)

{'reverse_url': 'https://nominatim.openstreetmap.org/reverse', 'overpass_url': 'https://overpass-api.de/api/interpreter', 'sleep_seconds': 1.0}


## 4.8 Build reverse geocoding function (backend-aware)

We create a reusable reverse geocoding function that works with both online and local backends and returns structured geo fields.

In [23]:
# [4.8] Backend-aware reverse geocoding function

import requests
import time

def reverse_geocode(lat, lon, geo_config):
    urls = get_geo_service_urls(geo_config)

    params = {
        "lat": lat,
        "lon": lon,
        "format": "json"
    }

    headers = {
        "User-Agent": geo_config["user_agent"]
    }

    try:
        response = requests.get(
            urls["reverse_url"],
            params=params,
            headers=headers,
            timeout=geo_config["request_timeout_seconds"]
        )

        data = response.json()
        address = data.get("address", {})

        result = {
            "city_desc": address.get("city") or address.get("town") or address.get("village"),
            "state_desc": address.get("state"),
            "county_desc": address.get("county"),
            "suburb_desc": address.get("suburb") or address.get("neighbourhood"),
            "postcode": address.get("postcode"),
            "country_desc": address.get("country"),
            "country_code": address.get("country_code"),
            "road_desc": address.get("road"),
            "display_name": data.get("display_name"),
            "osm_type": data.get("osm_type"),
            "osm_id": data.get("osm_id"),
            "geo_backend_code": geo_config["geo_backend_code"]
        }

    except:
        result = {
            "city_desc": None,
            "state_desc": None,
            "county_desc": None,
            "suburb_desc": None,
            "postcode": None,
            "country_desc": None,
            "country_code": None,
            "road_desc": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "geo_backend_code": geo_config["geo_backend_code"]
        }

    time.sleep(urls["sleep_seconds"])

    return result

## 4.9 Test reverse geocoding on one row

We test the reverse geocoding function on one valid coordinate row.

In [24]:
# [4.9] Test reverse geocoding on one valid row

sample_row = df[df["is_valid_coordinate"]].iloc[0]

geo_result = reverse_geocode(
    lat=sample_row["latitude_final"],
    lon=sample_row["longitude_final"],
    geo_config=GEO_CONFIG
)

geo_result

{'city_desc': 'Los Angeles',
 'state_desc': 'California',
 'county_desc': 'Los Angeles County',
 'suburb_desc': 'North Hollywood West Neighborhood Council District',
 'postcode': '91605',
 'country_desc': 'United States',
 'country_code': 'us',
 'road_desc': 'Beeman Avenue',
 'display_name': '7799, Beeman Avenue, North Hollywood West Neighborhood Council District, Los Angeles, Los Angeles County, California, 91605, United States',
 'osm_type': 'way',
 'osm_id': 13446744,
 'geo_backend_code': 'online'}

## 4.10 Define POI categories

We define the first POI categories and keep in mind that POI enrichment uses a snapshot in time.

In [25]:
# [4.10] Define POI categories for enrichment

POI_CONFIG = {
    "poi_radius_meters": 500,
    "poi_categories": {
        "police": [{"key": "amenity", "value": "police"}],
        "hospital": [{"key": "amenity", "value": "hospital"}],
        "school": [{"key": "amenity", "value": "school"}],
        "bank": [{"key": "amenity", "value": "bank"}],
        "bus_stop": [{"key": "highway", "value": "bus_stop"}],
        "train_station": [{"key": "railway", "value": "station"}],
        "metro_station": [{"key": "railway", "value": "station"}],
        "airport": [{"key": "aeroway", "value": "aerodrome"}],
        "place_of_worship": [{"key": "amenity", "value": "place_of_worship"}],
        "stadium": [{"key": "leisure", "value": "stadium"}],
        "sports_centre": [{"key": "leisure", "value": "sports_centre"}],
        "jewelry": [{"key": "shop", "value": "jewelry"}]
    }
}

print(POI_CONFIG)

{'poi_radius_meters': 500, 'poi_categories': {'police': [{'key': 'amenity', 'value': 'police'}], 'hospital': [{'key': 'amenity', 'value': 'hospital'}], 'school': [{'key': 'amenity', 'value': 'school'}], 'bank': [{'key': 'amenity', 'value': 'bank'}], 'bus_stop': [{'key': 'highway', 'value': 'bus_stop'}], 'train_station': [{'key': 'railway', 'value': 'station'}], 'metro_station': [{'key': 'railway', 'value': 'station'}], 'airport': [{'key': 'aeroway', 'value': 'aerodrome'}], 'place_of_worship': [{'key': 'amenity', 'value': 'place_of_worship'}], 'stadium': [{'key': 'leisure', 'value': 'stadium'}], 'sports_centre': [{'key': 'leisure', 'value': 'sports_centre'}], 'jewelry': [{'key': 'shop', 'value': 'jewelry'}]}}


## 4.11 Build Overpass query builder

We create a helper function to generate Overpass queries dynamically based on location, radius, and POI categories.

In [26]:
# [4.11] Build Overpass query builder

def build_overpass_query(lat, lon, radius, category_filters):
    query_parts = []

    for f in category_filters:
        key = f["key"]
        value = f["value"]

        part = f"""
        node["{key}"="{value}"](around:{radius},{lat},{lon});
        way["{key}"="{value}"](around:{radius},{lat},{lon});
        relation["{key}"="{value}"](around:{radius},{lat},{lon});
        """
        query_parts.append(part)

    full_query = f"""
    [out:json][timeout:25];
    (
        {"".join(query_parts)}
    );
    out center;
    """

    return full_query

## 4.12 Test Overpass query generation

We test the query builder to ensure it generates valid Overpass queries.

In [27]:
# [4.12] Test query builder

sample_lat = 34.2124
sample_lon = -118.4092

sample_category = POI_CONFIG["poi_categories"]["police"]

query = build_overpass_query(
    lat=sample_lat,
    lon=sample_lon,
    radius=POI_CONFIG["poi_radius_meters"],
    category_filters=sample_category
)

print(query[:500])  # print first part only


    [out:json][timeout:25];
    (
        
        node["amenity"="police"](around:500,34.2124,-118.4092);
        way["amenity"="police"](around:500,34.2124,-118.4092);
        relation["amenity"="police"](around:500,34.2124,-118.4092);
        
    );
    out center;
    


## 4.13 Execute Overpass query

We execute the Overpass query to retrieve nearby POIs from OpenStreetMap.

In [28]:
# [4.13] Execute Overpass query

def execute_overpass_query(query, geo_config):
    urls = get_geo_service_urls(geo_config)

    try:
        response = requests.post(
            urls["overpass_url"],
            data=query,
            timeout=geo_config["request_timeout_seconds"]
        )

        data = response.json()

        return data

    except:
        return {"elements": []}

## 4.14 Test Overpass API

We test retrieving nearby POIs for a single category (police).

In [29]:
# [4.14] Test Overpass execution

sample_lat = 34.2124
sample_lon = -118.4092

category_filters = POI_CONFIG["poi_categories"]["police"]

query = build_overpass_query(
    lat=sample_lat,
    lon=sample_lon,
    radius=POI_CONFIG["poi_radius_meters"],
    category_filters=category_filters
)

result = execute_overpass_query(query, GEO_CONFIG)

len(result.get("elements", []))

0

## 4.15 Test Overpass API with a dense location

We test Overpass using a dense urban location to confirm that POI retrieval works.

In [30]:
# [4.15] Test Overpass on a dense location with bus stops

test_lat = 34.0522
test_lon = -118.2437

category_filters = POI_CONFIG["poi_categories"]["bus_stop"]

query = build_overpass_query(
    lat=test_lat,
    lon=test_lon,
    radius=POI_CONFIG["poi_radius_meters"],
    category_filters=category_filters
)

result = execute_overpass_query(query, GEO_CONFIG)

print("POI count:", len(result.get("elements", [])))

POI count: 0


## 4.16 Parse Overpass POI results

We convert raw Overpass results into a clean dataframe with coordinates and POI category details.

In [31]:
# [4.16] Parse Overpass elements into a dataframe

def parse_overpass_elements(elements, category_name):
    parsed_rows = []

    for element in elements:
        if element["type"] == "node":
            poi_lat = element.get("lat")
            poi_lon = element.get("lon")
        else:
            center = element.get("center", {})
            poi_lat = center.get("lat")
            poi_lon = center.get("lon")

        tags = element.get("tags", {})

        parsed_rows.append({
            "osm_element_type": element.get("type"),
            "osm_id": element.get("id"),
            "poi_name": tags.get("name"),
            "poi_category_code": category_name,
            "poi_latitude": poi_lat,
            "poi_longitude": poi_lon
        })

    return pd.DataFrame(parsed_rows)

## 4.17 Preview parsed POI results

We parse the Overpass response and inspect the nearby POIs in tabular form.

In [32]:
# [4.17] Convert Overpass result to dataframe and preview

poi_df = parse_overpass_elements(
    result.get("elements", []),
    category_name="bus_stop"
)

print("Parsed POI rows:", len(poi_df))

poi_df.head(10)

Parsed POI rows: 0


""


## 4.18 Compute POI distance and coverage features

We calculate POI count, nearest real distance, and whether at least one POI exists within the search radius.

In [33]:
# [4.18] Compute distance-based POI features

import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius_m = 6371000

    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    delta_lat_rad = np.radians(lat2 - lat1)
    delta_lon_rad = np.radians(lon2 - lon1)

    a = (
        np.sin(delta_lat_rad / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon_rad / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return earth_radius_m * c


def compute_poi_features(poi_df, lat, lon, radius):
    if poi_df.empty:
        return {
            "poi_count": 0,
            "poi_exists_within_radius_flag": 0,
            "distance_to_nearest_poi_m": None
        }

    distances = poi_df.apply(
        lambda row: haversine_distance(
            lat,
            lon,
            row["poi_latitude"],
            row["poi_longitude"]
        ),
        axis=1
    )

    nearest_distance_m = float(distances.min())
    poi_count = int(len(poi_df))
    poi_exists_within_radius_flag = int(poi_count > 0)

    return {
        "poi_count": poi_count,
        "poi_exists_within_radius_flag": poi_exists_within_radius_flag,
        "distance_to_nearest_poi_m": nearest_distance_m
    }

## 4.19 Test POI feature computation

We compute POI count, nearest distance, and coverage flag for the sample location.

In [34]:
# [4.19] Test POI feature computation

features = compute_poi_features(
    poi_df=poi_df,
    lat=test_lat,
    lon=test_lon,
    radius=POI_CONFIG["poi_radius_meters"]
)

features

{'poi_count': 0,
 'poi_exists_within_radius_flag': 0,
 'distance_to_nearest_poi_m': None}

## 4.20 Enrich one point with all POI categories

We compute POI features for all configured categories for a single location.

In [35]:
# [4.20] Enrich one point with all POI categories

def enrich_poi_features_for_point(lat, lon, poi_config, geo_config):
    poi_features = {}

    for category_code, category_filters in poi_config["poi_categories"].items():
        query = build_overpass_query(
            lat=lat,
            lon=lon,
            radius=poi_config["poi_radius_meters"],
            category_filters=category_filters
        )

        result = execute_overpass_query(query, geo_config)

        poi_df = parse_overpass_elements(
            result.get("elements", []),
            category_name=category_code
        )

        category_features = compute_poi_features(
            poi_df=poi_df,
            lat=lat,
            lon=lon,
            radius=poi_config["poi_radius_meters"]
        )

        poi_features[f"poi_{category_code}_count_{poi_config['poi_radius_meters']}m"] = category_features["poi_count"]
        poi_features[f"poi_{category_code}_exists_{poi_config['poi_radius_meters']}m_flag"] = category_features["poi_exists_within_radius_flag"]
        poi_features[f"distance_to_nearest_{category_code}_m"] = category_features["distance_to_nearest_poi_m"]

    return poi_features

## 4.21 Test multi-category POI enrichment on one point

We test POI enrichment for one location across all configured categories.

In [36]:
# [4.21] Test POI enrichment for one point

poi_features = enrich_poi_features_for_point(
    lat=test_lat,
    lon=test_lon,
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG
)

poi_features

{'poi_police_count_500m': 0,
 'poi_police_exists_500m_flag': 0,
 'distance_to_nearest_police_m': None,
 'poi_hospital_count_500m': 0,
 'poi_hospital_exists_500m_flag': 0,
 'distance_to_nearest_hospital_m': None,
 'poi_school_count_500m': 0,
 'poi_school_exists_500m_flag': 0,
 'distance_to_nearest_school_m': None,
 'poi_bank_count_500m': 0,
 'poi_bank_exists_500m_flag': 0,
 'distance_to_nearest_bank_m': None,
 'poi_bus_stop_count_500m': 0,
 'poi_bus_stop_exists_500m_flag': 0,
 'distance_to_nearest_bus_stop_m': None,
 'poi_train_station_count_500m': 0,
 'poi_train_station_exists_500m_flag': 0,
 'distance_to_nearest_train_station_m': None,
 'poi_metro_station_count_500m': 0,
 'poi_metro_station_exists_500m_flag': 0,
 'distance_to_nearest_metro_station_m': None,
 'poi_airport_count_500m': 0,
 'poi_airport_exists_500m_flag': 0,
 'distance_to_nearest_airport_m': None,
 'poi_place_of_worship_count_500m': 0,
 'poi_place_of_worship_exists_500m_flag': 0,
 'distance_to_nearest_place_of_worship_m'

## 4.22 Apply POI enrichment to a small sample

We enrich a small set of valid rows to validate dataset-level POI enrichment before scaling.

In [37]:
# [4.22] Apply POI enrichment to a small sample

sample_df = df[df["is_valid_coordinate"]].head(3).copy()

poi_feature_rows = []

for _, row in sample_df.iterrows():
    poi_features = enrich_poi_features_for_point(
        lat=row["latitude_final"],
        lon=row["longitude_final"],
        poi_config=POI_CONFIG,
        geo_config=GEO_CONFIG
    )
    poi_feature_rows.append(poi_features)

sample_poi_features_df = pd.DataFrame(poi_feature_rows)

sample_poi_features_df

,poi_police_count_500m,poi_police_exists_500m_flag,distance_to_nearest_police_m,poi_hospital_count_500m,poi_hospital_exists_500m_flag,distance_to_nearest_hospital_m,poi_school_count_500m,poi_school_exists_500m_flag,distance_to_nearest_school_m,poi_bank_count_500m,...,distance_to_nearest_place_of_worship_m,poi_stadium_count_500m,poi_stadium_exists_500m_flag,distance_to_nearest_stadium_m,poi_sports_centre_count_500m,poi_sports_centre_exists_500m_flag,distance_to_nearest_sports_centre_m,poi_jewelry_count_500m,poi_jewelry_exists_500m_flag,distance_to_nearest_jewelry_m
0,0,0,None,0,0,None,0,0,None,0,...,None,0,0,None,0,0,None,0,0,None
1,0,0,None,0,0,None,0,0,None,0,...,None,0,0,None,0,0,None,0,0,None
2,0,0,None,0,0,None,0,0,None,0,...,None,0,0,None,0,0,None,0,0,None


## 4.23 Test enrichment on a dense location from dataset

We select a row with known active urban coordinates to validate POI enrichment.

In [38]:
# [4.23] Find a row near dense urban area (LA center)

dense_df = df[
    (df["latitude_final"].between(34.04, 34.06)) &
    (df["longitude_final"].between(-118.26, -118.23))
].head(1)

dense_df

,case_id,date_reported,date_occured,time_occured,area_code,area_name,report_district,crime_part,crime_code,crime_description,...,season,victim_age_group_code,victim_age_group_desc,latitude_original,longitude_original,is_valid_coordinate,latitude_final,longitude_final,geo_source,geo_precision_desc
563,210118909,2021-10-19,2020-06-02,16:05:00,1,Central,134,1,480,BIKE - STOLEN,...,summer,20_35,young_adult,34.051,-118.248,True,34.051,-118.248,original_coordinate,exact_coordinate


In [39]:
# [4.24] Apply POI enrichment on dense row

row = dense_df.iloc[0]

dense_features = enrich_poi_features_for_point(
    lat=row["latitude_final"],
    lon=row["longitude_final"],
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG
)

dense_features

{'poi_police_count_500m': 0,
 'poi_police_exists_500m_flag': 0,
 'distance_to_nearest_police_m': None,
 'poi_hospital_count_500m': 0,
 'poi_hospital_exists_500m_flag': 0,
 'distance_to_nearest_hospital_m': None,
 'poi_school_count_500m': 0,
 'poi_school_exists_500m_flag': 0,
 'distance_to_nearest_school_m': None,
 'poi_bank_count_500m': 0,
 'poi_bank_exists_500m_flag': 0,
 'distance_to_nearest_bank_m': None,
 'poi_bus_stop_count_500m': 0,
 'poi_bus_stop_exists_500m_flag': 0,
 'distance_to_nearest_bus_stop_m': None,
 'poi_train_station_count_500m': 0,
 'poi_train_station_exists_500m_flag': 0,
 'distance_to_nearest_train_station_m': None,
 'poi_metro_station_count_500m': 0,
 'poi_metro_station_exists_500m_flag': 0,
 'distance_to_nearest_metro_station_m': None,
 'poi_airport_count_500m': 0,
 'poi_airport_exists_500m_flag': 0,
 'distance_to_nearest_airport_m': None,
 'poi_place_of_worship_count_500m': 0,
 'poi_place_of_worship_exists_500m_flag': 0,
 'distance_to_nearest_place_of_worship_m'

## 4.25 Add POI cache layer

We introduce a cache to avoid repeated API calls for the same coordinates.

In [40]:
# [4.25] POI cache dictionary

poi_cache = {}

## 4.26 Cached POI enrichment function

We use caching to reuse results for repeated coordinates.

In [41]:
# [4.26] Enrich with cache

def enrich_poi_features_with_cache(lat, lon, poi_config, geo_config):
    key = (round(lat, 5), round(lon, 5))

    if key in poi_cache:
        return poi_cache[key]

    features = enrich_poi_features_for_point(
        lat=lat,
        lon=lon,
        poi_config=poi_config,
        geo_config=geo_config
    )

    poi_cache[key] = features

    return features

## 4.27 Test cache behavior

We verify that repeated calls reuse cached results.

In [42]:
# [4.27] Test cache

row = dense_df.iloc[0]

f1 = enrich_poi_features_with_cache(
    lat=row["latitude_final"],
    lon=row["longitude_final"],
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG
)

f2 = enrich_poi_features_with_cache(
    lat=row["latitude_final"],
    lon=row["longitude_final"],
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG
)

print("Cache size:", len(poi_cache))

Cache size: 1


## 4.28 Save and load POI cache

We add helper functions to save and reload the POI cache for checkpointing and resume.

In [43]:
# [4.28] Define cache save and load functions

import json
from pathlib import Path

def save_poi_cache(poi_cache, cache_path):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    serializable_cache = {
        f"{lat},{lon}": value
        for (lat, lon), value in poi_cache.items()
    }

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(serializable_cache, f, ensure_ascii=False, indent=2)


def load_poi_cache(cache_path):
    cache_path = Path(cache_path)

    if not cache_path.exists():
        return {}

    with open(cache_path, "r", encoding="utf-8") as f:
        serializable_cache = json.load(f)

    poi_cache = {}
    for key, value in serializable_cache.items():
        lat_str, lon_str = key.split(",")
        poi_cache[(float(lat_str), float(lon_str))] = value

    return poi_cache

## 4.29 Save and reload cache checkpoint

We test cache persistence by saving the current cache to disk and loading it back.

In [44]:
# [4.29] Test cache save and load

cache_file_path = "../data/interim/poi_cache_2.json"

save_poi_cache(poi_cache, cache_file_path)

loaded_poi_cache = load_poi_cache(cache_file_path)

print("Original cache size:", len(poi_cache))
print("Loaded cache size  :", len(loaded_poi_cache))

Original cache size: 1
Loaded cache size  : 1


## 4.30 Dataset enrichment with checkpointing

We enrich a small batch of dataset rows with POI features and save progress incrementally.

In [45]:
# [4.30] Enrich dataset with checkpointing (small batch)

output_file_path = "../data/interim/sample_poi_enriched2.csv"

batch_size = 5

sample_df = df[df["is_valid_coordinate"]].head(batch_size).copy()

results = []

# load cache if exists
poi_cache = load_poi_cache("../data/interim/poi_cache2.json")

for i, (_, row) in enumerate(sample_df.iterrows(), start=1):

    features = enrich_poi_features_with_cache(
        lat=row["latitude_final"],
        lon=row["longitude_final"],
        poi_config=POI_CONFIG,
        geo_config=GEO_CONFIG
    )

    combined_row = {
        **row.to_dict(),
        **features
    }

    results.append(combined_row)

    # save checkpoint every row (safe for now)
    pd.DataFrame(results).to_csv(output_file_path, index=False)

    # save cache
    save_poi_cache(poi_cache, "../data/interim/poi_cache2.json")

    print(f"Processed row {i}/{batch_size}")

print("Done.")

Processed row 1/5
Processed row 2/5
Processed row 3/5
Processed row 4/5
Processed row 5/5
Done.


## 4.31 Define POI enrichment metadata

We define metadata fields to make POI enrichment reproducible and traceable.

In [46]:
# [4.31] Define POI enrichment metadata

from datetime import datetime

POI_ENRICHMENT_CONFIG = {
    "poi_feature_version_code": "poi_v1_single_radius",
    "poi_temporal_mode_code": "current_snapshot",
    "poi_temporal_mode_desc": "Current OpenStreetMap snapshot used for enrichment",
    "poi_snapshot_date": datetime.utcnow().date().isoformat()
}

POI_ENRICHMENT_CONFIG

{'poi_feature_version_code': 'poi_v1_single_radius',
 'poi_temporal_mode_code': 'current_snapshot',
 'poi_temporal_mode_desc': 'Current OpenStreetMap snapshot used for enrichment',
 'poi_snapshot_date': '2026-04-30'}

## 4.32 Create POI cache key

We create a version-aware cache key so cached POI results remain reproducible.

In [47]:
# [4.32] Create version-aware POI cache key

def create_poi_cache_key(lat, lon, poi_config, poi_enrichment_config):
    return (
        round(lat, 5),
        round(lon, 5),
        poi_config["poi_radius_meters"],
        poi_enrichment_config["poi_feature_version_code"],
        poi_enrichment_config["poi_snapshot_date"]
    )

## 4.33 Update cached POI enrichment

We update the cache function to use the version-aware cache key.

In [48]:
# [4.33] Update cached POI enrichment function

def enrich_poi_features_with_cache(
    lat,
    lon,
    poi_config,
    geo_config,
    poi_enrichment_config
):
    key = create_poi_cache_key(
        lat=lat,
        lon=lon,
        poi_config=poi_config,
        poi_enrichment_config=poi_enrichment_config
    )

    if key in poi_cache:
        return poi_cache[key]

    features = enrich_poi_features_for_point(
        lat=lat,
        lon=lon,
        poi_config=poi_config,
        geo_config=geo_config
    )

    poi_cache[key] = features

    return features

## 4.34 Test updated POI cache function

We test the version-aware POI cache function on one dense location.

In [49]:
# [4.34] Test updated POI cache function

poi_cache = {}

row = dense_df.iloc[0]

test_features = enrich_poi_features_with_cache(
    lat=row["latitude_final"],
    lon=row["longitude_final"],
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG,
    poi_enrichment_config=POI_ENRICHMENT_CONFIG
)

print("Cache size:", len(poi_cache))

test_features

Cache size: 1


{'poi_police_count_500m': 0,
 'poi_police_exists_500m_flag': 0,
 'distance_to_nearest_police_m': None,
 'poi_hospital_count_500m': 0,
 'poi_hospital_exists_500m_flag': 0,
 'distance_to_nearest_hospital_m': None,
 'poi_school_count_500m': 0,
 'poi_school_exists_500m_flag': 0,
 'distance_to_nearest_school_m': None,
 'poi_bank_count_500m': 0,
 'poi_bank_exists_500m_flag': 0,
 'distance_to_nearest_bank_m': None,
 'poi_bus_stop_count_500m': 0,
 'poi_bus_stop_exists_500m_flag': 0,
 'distance_to_nearest_bus_stop_m': None,
 'poi_train_station_count_500m': 0,
 'poi_train_station_exists_500m_flag': 0,
 'distance_to_nearest_train_station_m': None,
 'poi_metro_station_count_500m': 0,
 'poi_metro_station_exists_500m_flag': 0,
 'distance_to_nearest_metro_station_m': None,
 'poi_airport_count_500m': 0,
 'poi_airport_exists_500m_flag': 0,
 'distance_to_nearest_airport_m': None,
 'poi_place_of_worship_count_500m': 0,
 'poi_place_of_worship_exists_500m_flag': 0,
 'distance_to_nearest_place_of_worship_m'

## 4.35 Define parquet checkpoint helpers

We add helper functions for saving and loading enrichment checkpoints in parquet format.

In [50]:
# [4.35] Define parquet checkpoint helper functions

from pathlib import Path


def save_checkpoint_parquet(df_checkpoint, checkpoint_path):
    # Ensure checkpoint folder exists
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save dataframe as parquet
    df_checkpoint.to_parquet(
        checkpoint_path,
        index=False
    )


def load_checkpoint_parquet(checkpoint_path):
    # Convert path object
    checkpoint_path = Path(checkpoint_path)

    # Return empty dataframe if no checkpoint yet
    if not checkpoint_path.exists():
        return pd.DataFrame()

    # Load existing checkpoint
    return pd.read_parquet(
        checkpoint_path
    )

## 4.36 Test parquet checkpoint

We test saving and loading a parquet checkpoint.

In [51]:
# [4.36] Test parquet checkpoint save and load

checkpoint_path = "../data/interim/sample_enrichment_checkpoint2.parquet"

# Save sample checkpoint
save_checkpoint_parquet(
    sample_df,
    checkpoint_path
)

# Load checkpoint
checkpoint_df = load_checkpoint_parquet(
    checkpoint_path
)

print("Checkpoint rows:", len(checkpoint_df))

Checkpoint rows: 5


## 4.37 Define processed-row detection

We define a helper function to identify rows already enriched in a checkpoint.

In [52]:
# [4.37] Define processed row detection function

def get_processed_case_ids(checkpoint_df):
    # Return empty set if no checkpoint exists
    if checkpoint_df.empty:
        return set()

    # Use case_id as resume key
    return set(
        checkpoint_df["case_id"].astype(str)
    )

## 4.38 Test processed-row detection

We test which case IDs are already saved in the checkpoint.

In [53]:
# [4.38] Test processed row detection

processed_case_ids = get_processed_case_ids(
    checkpoint_df
)

print("Processed case count:", len(processed_case_ids))

list(processed_case_ids)[:5]

Processed case count: 5


['211507896', '201516622', '240913563', '201418201', '210704711']

## 4.39 Build resumable sample batch

We select only rows that are not already available in the checkpoint.

In [54]:
# [4.39] Select rows not yet processed

batch_size = 5

# Load existing checkpoint
checkpoint_path = "../data/interim/sample_enrichment_checkpoint2.parquet"
checkpoint_df = load_checkpoint_parquet(checkpoint_path)

# Get processed case IDs
processed_case_ids = get_processed_case_ids(checkpoint_df)

# Select valid rows not already processed
pending_df = df[
    df["is_valid_coordinate"]
    & ~df["case_id"].astype(str).isin(processed_case_ids)
].head(batch_size).copy()

print("Pending rows:", len(pending_df))

pending_df[["case_id", "latitude_final", "longitude_final"]].head()

Pending rows: 5


,case_id,latitude_final,longitude_final
5,240412063,34.0830,-118.1678
6,240317069,34.0100,-118.2900
7,201115217,34.1107,-118.2589
8,241708596,34.2763,-118.5210
9,242113813,34.1493,-118.5886


## 4.40 Define enrichment status fields

We add modular status fields to track which enrichment stages have been completed.

In [55]:
# [4.40] Define enrichment status helper function

def add_enrichment_status_fields(row_dict):
    # Mark reverse geocoding status
    row_dict["reverse_geo_enriched_flag"] = 1

    # Mark POI enrichment status
    row_dict["poi_enriched_flag"] = 1

    # Store backend used for POI enrichment
    row_dict["poi_backend_code"] = GEO_CONFIG["geo_backend_code"]

    # Store POI feature version
    row_dict["poi_feature_version_code"] = POI_ENRICHMENT_CONFIG["poi_feature_version_code"]

    # Store POI snapshot date
    row_dict["poi_snapshot_date"] = POI_ENRICHMENT_CONFIG["poi_snapshot_date"]

    # Store POI temporal mode
    row_dict["poi_temporal_mode_code"] = POI_ENRICHMENT_CONFIG["poi_temporal_mode_code"]

    return row_dict

## 4.41 Test enrichment status fields

We test adding POI enrichment status metadata to one sample row.

In [56]:
# [4.41] Test enrichment status fields

sample_row_dict = pending_df.iloc[0].to_dict()

sample_row_dict = add_enrichment_status_fields(
    sample_row_dict
)

{
    "reverse_geo_enriched_flag": sample_row_dict["reverse_geo_enriched_flag"],
    "poi_enriched_flag": sample_row_dict["poi_enriched_flag"],
    "poi_backend_code": sample_row_dict["poi_backend_code"],
    "poi_feature_version_code": sample_row_dict["poi_feature_version_code"],
    "poi_snapshot_date": sample_row_dict["poi_snapshot_date"],
    "poi_temporal_mode_code": sample_row_dict["poi_temporal_mode_code"]
}

{'reverse_geo_enriched_flag': 1,
 'poi_enriched_flag': 1,
 'poi_backend_code': 'online',
 'poi_feature_version_code': 'poi_v1_single_radius',
 'poi_snapshot_date': '2026-04-30',
 'poi_temporal_mode_code': 'current_snapshot'}

## 4.42 Build resumable enrichment append function

We define a function to enrich pending rows and append them to the checkpoint.

In [57]:
# [4.42] Define resumable enrichment append function

def enrich_pending_rows(
    pending_df,
    checkpoint_df,
    poi_config,
    geo_config,
    poi_enrichment_config
):
    enriched_rows = []

    for _, row in pending_df.iterrows():

        # Compute POI features
        poi_features = enrich_poi_features_with_cache(
            lat=row["latitude_final"],
            lon=row["longitude_final"],
            poi_config=poi_config,
            geo_config=geo_config,
            poi_enrichment_config=poi_enrichment_config
        )

        # Merge row + features
        row_dict = {
            **row.to_dict(),
            **poi_features
        }

        # Add enrichment metadata
        row_dict = add_enrichment_status_fields(
            row_dict
        )

        enriched_rows.append(
            row_dict
        )

    new_rows_df = pd.DataFrame(
        enriched_rows
    )

    # Append to previous checkpoint
    combined_df = pd.concat(
        [checkpoint_df, new_rows_df],
        ignore_index=True
    )

    return combined_df

## 4.43 Run resumable enrichment append

We enrich the pending rows, append them to the checkpoint, and save the updated checkpoint.

In [58]:
# [4.43] Run resumable enrichment append

checkpoint_path = "../data/interim/sample_enrichment_checkpoint2.parquet"

# Load latest checkpoint
checkpoint_df = load_checkpoint_parquet(
    checkpoint_path
)

# Enrich pending rows and append to checkpoint
updated_checkpoint_df = enrich_pending_rows(
    pending_df=pending_df,
    checkpoint_df=checkpoint_df,
    poi_config=POI_CONFIG,
    geo_config=GEO_CONFIG,
    poi_enrichment_config=POI_ENRICHMENT_CONFIG
)

# Save updated checkpoint
save_checkpoint_parquet(
    updated_checkpoint_df,
    checkpoint_path
)

print("Previous checkpoint rows:", len(checkpoint_df))
print("Pending rows added       :", len(pending_df))
print("Updated checkpoint rows :", len(updated_checkpoint_df))

Previous checkpoint rows: 5
Pending rows added       : 5
Updated checkpoint rows : 10


## 4.44 Build unified geo enrichment function

We combine reverse geocoding and POI enrichment into one reusable geo enrichment function.

In [59]:
# [4.44] Define unified geo enrichment function

def enrich_geo_features_for_row(
    row,
    geo_config,
    poi_config,
    poi_enrichment_config
):
    # Run reverse geocoding
    reverse_geo_features = reverse_geocode(
        lat=row["latitude_final"],
        lon=row["longitude_final"],
        geo_config=geo_config
    )

    # Run POI enrichment with cache
    poi_features = enrich_poi_features_with_cache(
        lat=row["latitude_final"],
        lon=row["longitude_final"],
        poi_config=poi_config,
        geo_config=geo_config,
        poi_enrichment_config=poi_enrichment_config
    )

    # Combine all geo features
    geo_features = {
        **reverse_geo_features,
        **poi_features
    }

    # Add geo enrichment status
    geo_features["geo_enriched_flag"] = 1

    return geo_features

## 4.45 Test unified geo enrichment on one row

We test the unified geo enrichment function on a single valid row.

In [60]:
# [4.45] Test unified geo enrichment function

row = df[df["is_valid_coordinate"]].iloc[0]

geo_features = enrich_geo_features_for_row(
    row=row,
    geo_config=GEO_CONFIG,
    poi_config=POI_CONFIG,
    poi_enrichment_config=POI_ENRICHMENT_CONFIG
)

geo_features

{'city_desc': 'Los Angeles',
 'state_desc': 'California',
 'county_desc': 'Los Angeles County',
 'suburb_desc': 'North Hollywood West Neighborhood Council District',
 'postcode': '91605',
 'country_desc': 'United States',
 'country_code': 'us',
 'road_desc': 'Beeman Avenue',
 'display_name': '7799, Beeman Avenue, North Hollywood West Neighborhood Council District, Los Angeles, Los Angeles County, California, 91605, United States',
 'osm_type': 'way',
 'osm_id': 13446744,
 'geo_backend_code': 'online',
 'poi_police_count_500m': 0,
 'poi_police_exists_500m_flag': 0,
 'distance_to_nearest_police_m': None,
 'poi_hospital_count_500m': 0,
 'poi_hospital_exists_500m_flag': 0,
 'distance_to_nearest_hospital_m': None,
 'poi_school_count_500m': 0,
 'poi_school_exists_500m_flag': 0,
 'distance_to_nearest_school_m': None,
 'poi_bank_count_500m': 0,
 'poi_bank_exists_500m_flag': 0,
 'distance_to_nearest_bank_m': None,
 'poi_bus_stop_count_500m': 0,
 'poi_bus_stop_exists_500m_flag': 0,
 'distance_to

## 4.46 Build unified geo batch enrichment function

We update the batch enrichment logic to apply full geo enrichment to pending rows.

In [61]:
# [4.46] Define unified geo batch enrichment function

def enrich_pending_rows_with_geo(
    pending_df,
    checkpoint_df,
    geo_config,
    poi_config,
    poi_enrichment_config
):
    # Store enriched rows
    enriched_rows = []

    for _, row in pending_df.iterrows():

        # Run full geo enrichment
        geo_features = enrich_geo_features_for_row(
            row=row,
            geo_config=geo_config,
            poi_config=poi_config,
            poi_enrichment_config=poi_enrichment_config
        )

        # Merge source row with geo features
        row_dict = {
            **row.to_dict(),
            **geo_features
        }

        # Add enrichment metadata
        row_dict = add_enrichment_status_fields(
            row_dict
        )

        # Add row to result list
        enriched_rows.append(
            row_dict
        )

    # Convert new rows to dataframe
    new_rows_df = pd.DataFrame(
        enriched_rows
    )

    # Append new rows to checkpoint
    combined_df = pd.concat(
        [checkpoint_df, new_rows_df],
        ignore_index=True
    )

    return combined_df

## 4.47 Test unified geo batch enrichment

We test full geo enrichment on a very small pending batch and inspect the appended result.

In [62]:
# [4.47] Test unified geo batch enrichment

test_pending_df = pending_df.head(1).copy()

test_checkpoint_df = pd.DataFrame()

test_geo_checkpoint_df = enrich_pending_rows_with_geo(
    pending_df=test_pending_df,
    checkpoint_df=test_checkpoint_df,
    geo_config=GEO_CONFIG,
    poi_config=POI_CONFIG,
    poi_enrichment_config=POI_ENRICHMENT_CONFIG
)

print("Rows enriched:", len(test_geo_checkpoint_df))

test_geo_checkpoint_df[
    [
        "case_id",
        "city_desc",
        "country_code",
        "geo_enriched_flag",
        "poi_enriched_flag"
    ]
]

Rows enriched: 1


,case_id,city_desc,country_code,geo_enriched_flag,poi_enriched_flag
0,240412063,Los Angeles,us,1,1


## 4.48 Save unified geo checkpoint

We save the unified geo-enriched sample checkpoint in parquet format.

In [63]:
# [4.48] Save unified geo checkpoint

geo_checkpoint_path = "../data/interim/sample_geo_enrichment_checkpoint2.parquet"

save_checkpoint_parquet(
    test_geo_checkpoint_df,
    geo_checkpoint_path
)

loaded_geo_checkpoint_df = load_checkpoint_parquet(
    geo_checkpoint_path
)

print("Saved geo checkpoint rows:", len(loaded_geo_checkpoint_df))


Saved geo checkpoint rows: 1


## 5. Geo Enrichment Milestone Summary

In this section we built the first CRIMEX geo enrichment pipeline, including:

- coordinate validation and geo quality fields
- reverse geocoding enrichment using switchable online/local backends
- nearby POI enrichment using OpenStreetMap Overpass
- distance, count, and coverage POI features
- caching to avoid repeated API calls
- parquet checkpointing and resumable batch processing
- unified geo enrichment combining administrative geography and POI context
- enrichment provenance metadata for reproducibility

Current output supports both:
- feature-engineered dataset enrichment
- foundations for future relational / graph-style artifacts

### Next Section

Next we move to criminal behavior enrichment, where we will derive behavioral, severity, modus operandi, and offense intelligence features.

## 5. Criminal Behavior Enrichment

We start extracting behavior-related features from crime descriptions, weapon information, premise details, and modus operandi fields.

In [64]:
# [5.1] Inspect behavior-related source columns

behavior_columns = [
    "crime_description",
    "modus_operandi",
    "weapon_code",
    "weapon_description",
    "premise_code",
    "premise_description",
    "crime_part",
    "case_status_description"
]

df[behavior_columns].head(10)

,crime_description,modus_operandi,weapon_code,weapon_description,premise_code,premise_description,crime_part,case_status_description
0,THEFT OF IDENTITY,0377,NaN,NaN,501.0,SINGLE FAMILY DWELLING,2,Invest Cont
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,200.0,KNIFE WITH BLADE 6INCHES OR LESS,102.0,SIDEWALK,1,Invest Cont
2,THEFT OF IDENTITY,0377,NaN,NaN,501.0,SINGLE FAMILY DWELLING,2,Invest Cont
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,0344,NaN,NaN,101.0,STREET,1,Invest Cont
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,NaN,NaN,103.0,ALLEY,1,Invest Cont
5,THEFT OF IDENTITY,0100,NaN,NaN,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",2,Invest Cont
6,THEFT OF IDENTITY,0100 0928 1822,NaN,NaN,501.0,SINGLE FAMILY DWELLING,2,Invest Cont
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,1259 0448 0304 1822 0500 0522,500.0,UNKNOWN WEAPON/OTHER WEAPON,121.0,YARD (RESIDENTIAL/BUSINESS),2,Adult Other
8,THEFT OF IDENTITY,1501,NaN,NaN,501.0,SINGLE FAMILY DWELLING,2,Invest Cont
9,THEFT OF IDENTITY,0930,NaN,NaN,501.0,SINGLE FAMILY DWELLING,2,Invest Cont


## 5.2 Define crime behavior taxonomy

We define reusable rules to classify crime descriptions into family and severity groups.

In [65]:
# [5.2] Define crime behavior taxonomy function

def classify_crime_behavior(crime_description):
    # Handle missing descriptions
    if pd.isna(crime_description):
        return "unknown", "unknown", "unknown", "unknown"

    # Normalize text
    text = str(crime_description).lower()

    # Fraud / identity crimes
    if "identity" in text or "fraud" in text or "credit" in text:
        return "fraud_identity", "fraud or identity crime", "medium", "medium severity"

    # Violent crimes
    if "assault" in text or "battery" in text or "weapon" in text:
        return "violent", "violent crime", "high", "high severity"

    # Theft / property crimes
    if "theft" in text or "burglary" in text or "vehicle" in text or "robbery" in text:
        return "property", "property crime", "medium", "medium severity"

    # Child related crimes
    if "child" in text or "chld" in text:
        return "child_related", "child related crime", "high", "high severity"

    # Sexual crimes
    if "sex" in text or "rape" in text:
        return "sexual", "sexual crime", "high", "high severity"

    # Default
    return "other", "other crime", "low", "low severity"

## 5.3 Apply crime behavior taxonomy

We apply the crime behavior taxonomy to create family and severity fields.

In [66]:
# [5.3] Apply crime behavior taxonomy

df[
    [
        "crime_family_code",
        "crime_family_desc",
        "crime_severity_code",
        "crime_severity_desc"
    ]
] = df["crime_description"].apply(
    lambda x: pd.Series(classify_crime_behavior(x))
)

df[
    [
        "crime_description",
        "crime_family_code",
        "crime_family_desc",
        "crime_severity_code",
        "crime_severity_desc"
    ]
].head(10)

,crime_description,crime_family_code,crime_family_desc,crime_severity_code,crime_severity_desc
0,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent,violent crime,high,high severity
2,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,property,property crime,medium,medium severity
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),property,property crime,medium,medium severity
5,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity
6,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,child_related,child related crime,high,high severity
8,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity
9,THEFT OF IDENTITY,fraud_identity,fraud or identity crime,medium,medium severity


## 5.4 Derive weapon behavior indicators

We derive weapon-related behavioral risk indicators from weapon fields.

In [67]:
# [5.4] Define weapon behavior indicator function

def derive_weapon_features(
    weapon_code,
    weapon_description
):
    # Default flags
    weapon_involved_flag = 0
    deadly_weapon_flag = 0
    unknown_weapon_flag = 0

    # If weapon code exists, weapon involved
    if pd.notna(weapon_code):
        weapon_involved_flag = 1

    # Normalize weapon description
    weapon_text = str(weapon_description).lower()

    # Deadly weapon indicators
    if (
        "knife" in weapon_text
        or "gun" in weapon_text
        or "firearm" in weapon_text
    ):
        deadly_weapon_flag = 1

    # Unknown weapon indicator
    if "unknown" in weapon_text:
        unknown_weapon_flag = 1

    return (
        weapon_involved_flag,
        deadly_weapon_flag,
        unknown_weapon_flag
    )

## 5.5 Apply weapon behavior indicators

We apply weapon behavior rules to create weapon-related risk flags.

In [68]:
# [5.5] Apply weapon behavior indicators

df[
    [
        "weapon_involved_flag",
        "deadly_weapon_flag",
        "unknown_weapon_flag"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_weapon_features(
            row["weapon_code"],
            row["weapon_description"]
        )
    ),
    axis=1
)

df[
    [
        "weapon_description",
        "weapon_involved_flag",
        "deadly_weapon_flag",
        "unknown_weapon_flag"
    ]
].head(10)

,weapon_description,weapon_involved_flag,deadly_weapon_flag,unknown_weapon_flag
0,NaN,0,0,0
1,KNIFE WITH BLADE 6INCHES OR LESS,1,1,0
2,NaN,0,0,0
3,NaN,0,0,0
4,NaN,0,0,0
5,NaN,0,0,0
6,NaN,0,0,0
7,UNKNOWN WEAPON/OTHER WEAPON,1,0,1
8,NaN,0,0,0
9,NaN,0,0,0


## 5.6 Derive modus operandi complexity features

We derive behavioral complexity features from modus operandi codes.

In [69]:
# [5.6] Define modus operandi feature function

def derive_mo_features(modus_operandi):
    # Handle missing MO
    if pd.isna(modus_operandi):
        return (
            0,
            0,
            "none",
            "no modus operandi"
        )

    # Split MO codes
    mo_codes = str(modus_operandi).split()

    mo_code_count = len(mo_codes)

    # Multi-step indicator
    multi_step_behavior_flag = int(
        mo_code_count > 1
    )

    # Simple complexity grouping
    if mo_code_count == 1:
        mo_complexity_code = "simple"
        mo_complexity_desc = "single-step behavior"

    elif mo_code_count <= 3:
        mo_complexity_code = "moderate"
        mo_complexity_desc = "moderately complex behavior"

    else:
        mo_complexity_code = "complex"
        mo_complexity_desc = "multi-step complex behavior"

    return (
        mo_code_count,
        multi_step_behavior_flag,
        mo_complexity_code,
        mo_complexity_desc
    )

## 5.7 Apply modus operandi complexity features

We apply modus operandi rules to create behavioral complexity features.

In [70]:
# [5.7] Apply modus operandi complexity features

df[
    [
        "mo_code_count",
        "multi_step_behavior_flag",
        "mo_complexity_code",
        "mo_complexity_desc"
    ]
] = df["modus_operandi"].apply(
    lambda x: pd.Series(
        derive_mo_features(x)
    )
)

df[
    [
        "modus_operandi",
        "mo_code_count",
        "multi_step_behavior_flag",
        "mo_complexity_code",
        "mo_complexity_desc"
    ]
].head(10)

,modus_operandi,mo_code_count,multi_step_behavior_flag,mo_complexity_code,mo_complexity_desc
0,0377,1,0,simple,single-step behavior
1,0416 0334 2004 1822 1414 0305 0319 0400,8,1,complex,multi-step complex behavior
2,0377,1,0,simple,single-step behavior
3,0344,1,0,simple,single-step behavior
4,1300 0344 1606 2032,4,1,complex,multi-step complex behavior
5,0100,1,0,simple,single-step behavior
6,0100 0928 1822,3,1,moderate,moderately complex behavior
7,1259 0448 0304 1822 0500 0522,6,1,complex,multi-step complex behavior
8,1501,1,0,simple,single-step behavior
9,0930,1,0,simple,single-step behavior


## 5.8 Create target context features

We derive basic target context from crime description and premise description.

In [71]:
# [5.8] Define target context feature function

def derive_target_context(
    crime_description,
    premise_description
):
    # Normalize text
    crime_text = str(crime_description).lower()
    premise_text = str(premise_description).lower()

    # Identity target
    if "identity" in crime_text:
        return "identity", "identity-related target"

    # Vehicle target
    if "vehicle" in crime_text:
        return "vehicle", "vehicle-related target"

    # Child target
    if "child" in crime_text or "chld" in crime_text:
        return "child", "child-related target"

    # Residential target
    if "dwelling" in premise_text or "apartment" in premise_text:
        return "residential", "residential target"

    # Public space target
    if "street" in premise_text or "sidewalk" in premise_text or "alley" in premise_text:
        return "public_space", "public space target"

    return "other", "other target context"

## 5.9 Apply target context features

We apply target context rules to classify what the crime appears to target.

In [72]:
# [5.9] Apply target context features

df[
    [
        "target_context_code",
        "target_context_desc"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_target_context(
            row["crime_description"],
            row["premise_description"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "premise_description",
        "target_context_code",
        "target_context_desc"
    ]
].head(10)

,crime_description,premise_description,target_context_code,target_context_desc
0,THEFT OF IDENTITY,SINGLE FAMILY DWELLING,identity,identity-related target
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",SIDEWALK,public_space,public space target
2,THEFT OF IDENTITY,SINGLE FAMILY DWELLING,identity,identity-related target
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,STREET,vehicle,vehicle-related target
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),ALLEY,vehicle,vehicle-related target
5,THEFT OF IDENTITY,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",identity,identity-related target
6,THEFT OF IDENTITY,SINGLE FAMILY DWELLING,identity,identity-related target
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,YARD (RESIDENTIAL/BUSINESS),child,child-related target
8,THEFT OF IDENTITY,SINGLE FAMILY DWELLING,identity,identity-related target
9,THEFT OF IDENTITY,SINGLE FAMILY DWELLING,identity,identity-related target


## 5.10 Check behavior feature distributions

We check the distribution of the new behavior features.

In [73]:
# [5.10] Check behavior feature distributions

behavior_summary = {
    "crime_family": df["crime_family_code"].value_counts(),
    "crime_severity": df["crime_severity_code"].value_counts(),
    "mo_complexity": df["mo_complexity_code"].value_counts(),
    "target_context": df["target_context_code"].value_counts()
}

behavior_summary

{'crime_family': crime_family_code
 property          508522
 violent           238043
 other             181832
 fraud_identity     63166
 sexual              8467
 child_related       4864
 Name: count, dtype: int64,
 'crime_severity': crime_severity_code
 medium    571688
 high      251374
 low       181832
 Name: count, dtype: int64,
 'mo_complexity': mo_complexity_code
 moderate    359771
 complex     352536
 none        151598
 simple      140989
 Name: count, dtype: int64,
 'target_context': target_context_code
 other           311244
 vehicle         266575
 residential     219703
 public_space    135800
 identity         62536
 child             9036
 Name: count, dtype: int64}

## 5.11 Define behavior risk profile

We define composite behavioral risk profiles by combining crime family, victim context, weapon indicators, and modus operandi complexity.

In [74]:
# [5.11] Define behavior risk profile function

def derive_behavior_risk_profile(
    crime_family_code,
    deadly_weapon_flag,
    multi_step_behavior_flag,
    mo_complexity_code,
    target_context_code
):
    # High-risk violent behavior has highest priority
    if (
        crime_family_code == "violent"
        and deadly_weapon_flag == 1
    ):
        return (
            "violent_high_risk",
            "high risk violent behavior"
        )

    # Vulnerable target has high priority
    if target_context_code == "child":
        return (
            "vulnerable_targeted",
            "crime targeting vulnerable victims"
        )

    # Organized or complex behavior
    if (
        multi_step_behavior_flag == 1
        and mo_complexity_code == "complex"
    ):
        return (
            "organized_complex",
            "organized or complex behavior"
        )

    # Fraud planning behavior
    if crime_family_code == "fraud_identity":
        return (
            "structured_fraud",
            "structured fraud behavior"
        )

    # Property opportunistic behavior
    if crime_family_code == "property":
        return (
            "opportunistic_property",
            "opportunistic property behavior"
        )

    return (
        "general_risk",
        "general risk behavior"
    )

## 5.12 Apply behavior risk profile

We apply composite behavioral risk profiling using the engineered behavior features.

In [75]:
# [5.12] Apply behavior risk profile

df[
    [
        "behavior_risk_profile_code",
        "behavior_risk_profile_desc"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_behavior_risk_profile(
            row["crime_family_code"],
            row["deadly_weapon_flag"],
            row["multi_step_behavior_flag"],
            row["mo_complexity_code"],
            row["target_context_code"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "crime_family_code",
        "mo_complexity_code",
        "behavior_risk_profile_code",
        "behavior_risk_profile_desc"
    ]
].head(12)

,crime_description,crime_family_code,mo_complexity_code,behavior_risk_profile_code,behavior_risk_profile_desc
0,THEFT OF IDENTITY,fraud_identity,simple,structured_fraud,structured fraud behavior
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent,complex,violent_high_risk,high risk violent behavior
2,THEFT OF IDENTITY,fraud_identity,simple,structured_fraud,structured fraud behavior
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,property,simple,opportunistic_property,opportunistic property behavior
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),property,complex,organized_complex,organized or complex behavior
5,THEFT OF IDENTITY,fraud_identity,simple,structured_fraud,structured fraud behavior
6,THEFT OF IDENTITY,fraud_identity,moderate,structured_fraud,structured fraud behavior
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,child_related,complex,vulnerable_targeted,crime targeting vulnerable victims
8,THEFT OF IDENTITY,fraud_identity,simple,structured_fraud,structured fraud behavior
9,THEFT OF IDENTITY,fraud_identity,simple,structured_fraud,structured fraud behavior


## 5.13 Check behavior risk profile distribution

We review the distribution of the composite behavior risk profiles.

In [76]:
# [5.13] Check behavior risk profile distribution

df["behavior_risk_profile_code"].value_counts()

behavior_risk_profile_code
opportunistic_property    371859
organized_complex         320893
general_risk              214609
structured_fraud           56490
violent_high_risk          32062
vulnerable_targeted         8981
Name: count, dtype: int64

## 5.14 Define criminal sophistication score

We define a behavioral sophistication score using existing behavior features.

In [77]:
# [5.14] Define criminal sophistication score function

def derive_criminal_sophistication(
    multi_step_behavior_flag,
    mo_complexity_code,
    deadly_weapon_flag,
    crime_family_code
):
    sophistication_score = 0

    # Multi-step behavior adds sophistication
    if multi_step_behavior_flag == 1:
        sophistication_score += 1

    # Complex MO adds more sophistication
    if mo_complexity_code == "complex":
        sophistication_score += 2

    # Deadly weapon adds risk sophistication
    if deadly_weapon_flag == 1:
        sophistication_score += 1

    # Structured fraud often involves planning
    if crime_family_code == "fraud_identity":
        sophistication_score += 1

    # Map score to level
    if sophistication_score <= 1:
        sophistication_level_code = "low"

    elif sophistication_score <= 3:
        sophistication_level_code = "medium"

    else:
        sophistication_level_code = "high"

    return (
        sophistication_score,
        sophistication_level_code
    )

## 5.15 Apply criminal sophistication score

We apply the sophistication score function to create numeric and categorical behavior features.

In [78]:
# [5.15] Apply criminal sophistication score

df[
    [
        "criminal_sophistication_score",
        "criminal_sophistication_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_criminal_sophistication(
            row["multi_step_behavior_flag"],
            row["mo_complexity_code"],
            row["deadly_weapon_flag"],
            row["crime_family_code"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "mo_complexity_code",
        "deadly_weapon_flag",
        "crime_family_code",
        "criminal_sophistication_score",
        "criminal_sophistication_code"
    ]
].head(12)

,crime_description,mo_complexity_code,deadly_weapon_flag,crime_family_code,criminal_sophistication_score,criminal_sophistication_code
0,THEFT OF IDENTITY,simple,0,fraud_identity,1,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",complex,1,violent,4,high
2,THEFT OF IDENTITY,simple,0,fraud_identity,1,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,simple,0,property,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),complex,0,property,3,medium
5,THEFT OF IDENTITY,simple,0,fraud_identity,1,low
6,THEFT OF IDENTITY,moderate,0,fraud_identity,2,medium
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,complex,0,child_related,3,medium
8,THEFT OF IDENTITY,simple,0,fraud_identity,1,low
9,THEFT OF IDENTITY,simple,0,fraud_identity,1,low


## 5.16 Define behavioral signature risk score

We define a composite behavioral risk score distinct from sophistication.

In [79]:
# [5.16] Define behavioral signature risk function

def derive_behavior_signature_risk(
    behavior_risk_profile_code,
    deadly_weapon_flag,
    criminal_sophistication_score,
    target_context_code
):
    risk_score = 0

    # Profile-driven risk
    if behavior_risk_profile_code == "violent_high_risk":
        risk_score += 4

    elif behavior_risk_profile_code == "vulnerable_targeted":
        risk_score += 3

    elif behavior_risk_profile_code == "organized_complex":
        risk_score += 2

    elif behavior_risk_profile_code == "structured_fraud":
        risk_score += 1

    # Weapon contribution
    if deadly_weapon_flag == 1:
        risk_score += 2

    # Sophistication contribution
    risk_score += min(
        criminal_sophistication_score,
        3
    )

    # Vulnerable target boost
    if target_context_code == "child":
        risk_score += 2

    # Risk level mapping
    if risk_score <= 2:
        risk_level_code = "low"

    elif risk_score <= 5:
        risk_level_code = "medium"

    else:
        risk_level_code = "high"

    return (
        risk_score,
        risk_level_code
    )

## 5.17 Apply behavioral signature risk score

We apply the behavioral signature risk function to create numeric and categorical risk features.

In [80]:
# [5.17] Apply behavioral signature risk score

df[
    [
        "behavior_signature_risk_score",
        "behavior_signature_risk_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_behavior_signature_risk(
            row["behavior_risk_profile_code"],
            row["deadly_weapon_flag"],
            row["criminal_sophistication_score"],
            row["target_context_code"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_risk_profile_code",
        "criminal_sophistication_score",
        "target_context_code",
        "behavior_signature_risk_score",
        "behavior_signature_risk_code"
    ]
].head(12)

,crime_description,behavior_risk_profile_code,criminal_sophistication_score,target_context_code,behavior_signature_risk_score,behavior_signature_risk_code
0,THEFT OF IDENTITY,structured_fraud,1,identity,2,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent_high_risk,4,public_space,9,high
2,THEFT OF IDENTITY,structured_fraud,1,identity,2,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,opportunistic_property,0,vehicle,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),organized_complex,3,vehicle,5,medium
5,THEFT OF IDENTITY,structured_fraud,1,identity,2,low
6,THEFT OF IDENTITY,structured_fraud,2,identity,3,medium
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,vulnerable_targeted,3,child,8,high
8,THEFT OF IDENTITY,structured_fraud,1,identity,2,low
9,THEFT OF IDENTITY,structured_fraud,1,identity,2,low


## 5.18 Check behavioral risk distributions

We review the distribution of behavioral risk and sophistication features.

In [81]:
# [5.18] Check behavioral risk distributions

risk_summary = {
    "behavior_signature_risk": df["behavior_signature_risk_code"].value_counts(),
    "criminal_sophistication": df["criminal_sophistication_code"].value_counts(),
    "behavior_risk_profile": df["behavior_risk_profile_code"].value_counts()
}

risk_summary

{'behavior_signature_risk': behavior_signature_risk_code
 low       604787
 medium    345295
 high       54812
 Name: count, dtype: int64,
 'criminal_sophistication': criminal_sophistication_code
 low       608880
 medium    348672
 high       47342
 Name: count, dtype: int64,
 'behavior_risk_profile': behavior_risk_profile_code
 opportunistic_property    371859
 organized_complex         320893
 general_risk              214609
 structured_fraud           56490
 violent_high_risk          32062
 vulnerable_targeted         8981
 Name: count, dtype: int64}

## 5.19 Build behavior signature text

We create a normalized behavior signature text field for future semantic enrichment and embeddings.

In [82]:
# [5.19] Define behavior signature text function

def build_behavior_signature_text(
    crime_family_desc,
    weapon_description,
    target_context_desc,
    behavior_risk_profile_desc,
    mo_complexity_desc
):
    parts = []

    # Crime family
    if pd.notna(crime_family_desc):
        parts.append(
            str(crime_family_desc).lower()
        )

    # Weapon
    if pd.notna(weapon_description):
        parts.append(
            str(weapon_description).lower()
        )

    # Target context
    if pd.notna(target_context_desc):
        parts.append(
            str(target_context_desc).lower()
        )

    # Behavior profile
    if pd.notna(behavior_risk_profile_desc):
        parts.append(
            str(behavior_risk_profile_desc).lower()
        )

    # MO complexity
    if pd.notna(mo_complexity_desc):
        parts.append(
            str(mo_complexity_desc).lower()
        )

    return " | ".join(parts)

## 5.20 Apply behavior signature text

We create a behavior signature text field for semantic enrichment readiness.

In [83]:
# [5.20] Apply behavior signature text

df["behavior_signature_text"] = df.apply(
    lambda row: build_behavior_signature_text(
        row["crime_family_desc"],
        row["weapon_description"],
        row["target_context_desc"],
        row["behavior_risk_profile_desc"],
        row["mo_complexity_desc"]
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_signature_text"
    ]
].head(10)

,crime_description,behavior_signature_text
0,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent crime | knife with blade 6inches or le...
2,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,property crime | vehicle-related target | oppo...
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),property crime | vehicle-related target | orga...
5,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...
6,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,child related crime | unknown weapon/other wea...
8,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...
9,THEFT OF IDENTITY,fraud or identity crime | identity-related tar...


## 5.21 Create behavior signature key

We create a stable behavioral fingerprint key for grouping similar crime behavior patterns.

In [84]:
# [5.21] Define behavior signature key function

def build_behavior_signature_key(
    crime_family_code,
    target_context_code,
    behavior_risk_profile_code,
    mo_complexity_code,
    deadly_weapon_flag
):
    # Combine stable behavior components
    key_parts = [
        str(crime_family_code),
        str(target_context_code),
        str(behavior_risk_profile_code),
        str(mo_complexity_code),
        str(deadly_weapon_flag)
    ]

    # Return pipe-separated signature key
    return "|".join(key_parts)

## 5.22 Apply behavior signature key

We create a stable behavioral fingerprint key for each crime record.

In [85]:
# [5.22] Apply behavior signature key

df["behavior_signature_key"] = df.apply(
    lambda row: build_behavior_signature_key(
        row["crime_family_code"],
        row["target_context_code"],
        row["behavior_risk_profile_code"],
        row["mo_complexity_code"],
        row["deadly_weapon_flag"]
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_signature_key"
    ]
].head(10)

,crime_description,behavior_signature_key
0,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|simple|0
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent|public_space|violent_high_risk|complex|1
2,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|simple|0
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,property|vehicle|opportunistic_property|simple|0
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),property|vehicle|organized_complex|complex|0
5,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|simple|0
6,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|moder...
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,child_related|child|vulnerable_targeted|complex|0
8,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|simple|0
9,THEFT OF IDENTITY,fraud_identity|identity|structured_fraud|simple|0


## 5.23 Derive behavior signature rarity

We derive frequency and rarity indicators for behavioral fingerprints.

In [86]:
# [5.23] Compute behavior signature frequency and rarity

# Frequency of each signature
signature_counts = df["behavior_signature_key"].value_counts()

# Map frequency back to rows
df["behavior_signature_frequency"] = df[
    "behavior_signature_key"
].map(
    signature_counts
)

def classify_signature_rarity(signature_frequency):
    # Rare behavior signatures
    if signature_frequency <= 20:
        return "rare"

    # Uncommon signatures
    elif signature_frequency <= 500:
        return "uncommon"

    # Common signatures
    return "common"


df["behavior_signature_rarity_code"] = df[
    "behavior_signature_frequency"
].apply(
    classify_signature_rarity
)

df[
    [
        "behavior_signature_key",
        "behavior_signature_frequency",
        "behavior_signature_rarity_code"
    ]
].head(10)

,behavior_signature_key,behavior_signature_frequency,behavior_signature_rarity_code
0,fraud_identity|identity|structured_fraud|simple|0,19587,common
1,violent|public_space|violent_high_risk|complex|1,12921,common
2,fraud_identity|identity|structured_fraud|simple|0,19587,common
3,property|vehicle|opportunistic_property|simple|0,16802,common
4,property|vehicle|organized_complex|complex|0,46707,common
5,fraud_identity|identity|structured_fraud|simple|0,19587,common
6,fraud_identity|identity|structured_fraud|moder...,34165,common
7,child_related|child|vulnerable_targeted|complex|0,2549,common
8,fraud_identity|identity|structured_fraud|simple|0,19587,common
9,fraud_identity|identity|structured_fraud|simple|0,19587,common


## 5.24 Define escalation potential

We derive escalation potential from violence, weapons, vulnerability, and behavioral risk.

In [87]:
# [5.24] Define escalation potential function

def derive_escalation_potential(
    behavior_risk_profile_code,
    deadly_weapon_flag,
    criminal_sophistication_score,
    target_context_code,
    behavior_signature_risk_code
):
    escalation_score = 0

    # Violent risk
    if behavior_risk_profile_code == "violent_high_risk":
        escalation_score += 3

    # Vulnerable target
    if target_context_code == "child":
        escalation_score += 2

    # Weapon escalation
    if deadly_weapon_flag == 1:
        escalation_score += 2

    # Sophistication contribution
    if criminal_sophistication_score >= 3:
        escalation_score += 1

    # High behavior signature risk
    if behavior_signature_risk_code == "high":
        escalation_score += 2

    # Map escalation level
    if escalation_score <= 2:
        escalation_code = "low"

    elif escalation_score <= 5:
        escalation_code = "medium"

    else:
        escalation_code = "high"

    return (
        escalation_score,
        escalation_code
    )

## 5.25 Apply escalation potential

We apply escalation potential scoring to derive escalation risk features.

In [88]:
# [5.25] Apply escalation potential

df[
    [
        "escalation_potential_score",
        "escalation_potential_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_escalation_potential(
            row["behavior_risk_profile_code"],
            row["deadly_weapon_flag"],
            row["criminal_sophistication_score"],
            row["target_context_code"],
            row["behavior_signature_risk_code"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_risk_profile_code",
        "behavior_signature_risk_code",
        "escalation_potential_score",
        "escalation_potential_code"
    ]
].head(12)

,crime_description,behavior_risk_profile_code,behavior_signature_risk_code,escalation_potential_score,escalation_potential_code
0,THEFT OF IDENTITY,structured_fraud,low,0,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent_high_risk,high,8,high
2,THEFT OF IDENTITY,structured_fraud,low,0,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,opportunistic_property,low,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),organized_complex,medium,1,low
5,THEFT OF IDENTITY,structured_fraud,low,0,low
6,THEFT OF IDENTITY,structured_fraud,medium,0,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,vulnerable_targeted,high,5,medium
8,THEFT OF IDENTITY,structured_fraud,low,0,low
9,THEFT OF IDENTITY,structured_fraud,low,0,low


## 5.26 Define behavioral diversity

We derive a behavioral diversity indicator to capture adaptive or unusual crime behavior.

In [89]:
# [5.26] Define behavioral diversity function

def derive_behavioral_diversity(
    mo_complexity_code,
    behavior_signature_rarity_code,
    criminal_sophistication_score
):
    diversity_score = 0

    # Complex behavior contributes
    if mo_complexity_code == "complex":
        diversity_score += 2

    elif mo_complexity_code == "moderate":
        diversity_score += 1

    # Rare signatures suggest novelty
    if behavior_signature_rarity_code == "rare":
        diversity_score += 2

    elif behavior_signature_rarity_code == "uncommon":
        diversity_score += 1

    # Sophistication contributes
    if criminal_sophistication_score >= 3:
        diversity_score += 1

    # Diversity level mapping
    if diversity_score <= 1:
        diversity_code = "low"

    elif diversity_score <= 3:
        diversity_code = "medium"

    else:
        diversity_code = "high"

    return (
        diversity_score,
        diversity_code
    )

## 5.27 Apply behavioral diversity

We apply behavioral diversity scoring to derive adaptive or unusual behavior indicators.

In [90]:
# [5.27] Apply behavioral diversity

df[
    [
        "behavioral_diversity_score",
        "behavioral_diversity_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_behavioral_diversity(
            row["mo_complexity_code"],
            row["behavior_signature_rarity_code"],
            row["criminal_sophistication_score"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "mo_complexity_code",
        "behavior_signature_rarity_code",
        "criminal_sophistication_score",
        "behavioral_diversity_score",
        "behavioral_diversity_code"
    ]
].head(12)

,crime_description,mo_complexity_code,behavior_signature_rarity_code,criminal_sophistication_score,behavioral_diversity_score,behavioral_diversity_code
0,THEFT OF IDENTITY,simple,common,1,0,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",complex,common,4,3,medium
2,THEFT OF IDENTITY,simple,common,1,0,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,simple,common,0,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),complex,common,3,3,medium
5,THEFT OF IDENTITY,simple,common,1,0,low
6,THEFT OF IDENTITY,moderate,common,2,1,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,complex,common,3,3,medium
8,THEFT OF IDENTITY,simple,common,1,0,low
9,THEFT OF IDENTITY,simple,common,1,0,low


## 5.28 Define behavioral threat score

We derive a composite behavioral threat score from risk, escalation, and behavioral diversity.

In [91]:
# [5.28] Define behavioral threat score function

def derive_behavioral_threat(
    behavior_signature_risk_score,
    escalation_potential_score,
    behavioral_diversity_score
):
    # Composite threat score
    threat_score = (
        behavior_signature_risk_score
        + escalation_potential_score
        + behavioral_diversity_score
    )

    # Threat level mapping
    if threat_score <= 4:
        threat_code = "low"

    elif threat_score <= 8:
        threat_code = "medium"

    else:
        threat_code = "high"

    return (
        threat_score,
        threat_code
    )

## 5.29 Apply behavioral threat score

We apply the composite behavioral threat scoring function.

In [92]:
# [5.29] Apply behavioral threat score

df[
    [
        "behavioral_threat_score",
        "behavioral_threat_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_behavioral_threat(
            row["behavior_signature_risk_score"],
            row["escalation_potential_score"],
            row["behavioral_diversity_score"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_signature_risk_score",
        "escalation_potential_score",
        "behavioral_diversity_score",
        "behavioral_threat_score",
        "behavioral_threat_code"
    ]
].head(12)

,crime_description,behavior_signature_risk_score,escalation_potential_score,behavioral_diversity_score,behavioral_threat_score,behavioral_threat_code
0,THEFT OF IDENTITY,2,0,0,2,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",9,8,3,20,high
2,THEFT OF IDENTITY,2,0,0,2,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,0,0,0,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),5,1,3,9,high
5,THEFT OF IDENTITY,2,0,0,2,low
6,THEFT OF IDENTITY,3,0,1,4,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,8,5,3,16,high
8,THEFT OF IDENTITY,2,0,0,2,low
9,THEFT OF IDENTITY,2,0,0,2,low


## 5.30 Define offender persistence proxy

We derive a proxy indicator for persistent or repeat-pattern offender behavior.

In [93]:
# [5.30] Define offender persistence proxy function

def derive_offender_persistence_proxy(
    mo_complexity_code,
    behavior_signature_rarity_code,
    criminal_sophistication_score,
    behavior_risk_profile_code
):
    persistence_score = 0

    # Complex organized behavior
    if behavior_risk_profile_code == "organized_complex":
        persistence_score += 2

    # Rare or uncommon signatures
    if behavior_signature_rarity_code == "rare":
        persistence_score += 2

    elif behavior_signature_rarity_code == "uncommon":
        persistence_score += 1

    # Sophistication contribution
    if criminal_sophistication_score >= 3:
        persistence_score += 1

    # Complex MO
    if mo_complexity_code == "complex":
        persistence_score += 1

    # Persistence level
    if persistence_score <= 1:
        persistence_code = "low"

    elif persistence_score <= 3:
        persistence_code = "medium"

    else:
        persistence_code = "high"

    return (
        persistence_score,
        persistence_code
    )

## 5.31 Apply offender persistence proxy

We apply the persistence proxy function to estimate repeat-pattern behavioral tendency.

In [94]:
# [5.31] Apply offender persistence proxy

df[
    [
        "offender_persistence_proxy_score",
        "offender_persistence_proxy_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_offender_persistence_proxy(
            row["mo_complexity_code"],
            row["behavior_signature_rarity_code"],
            row["criminal_sophistication_score"],
            row["behavior_risk_profile_code"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_risk_profile_code",
        "mo_complexity_code",
        "behavior_signature_rarity_code",
        "offender_persistence_proxy_score",
        "offender_persistence_proxy_code"
    ]
].head(12)

,crime_description,behavior_risk_profile_code,mo_complexity_code,behavior_signature_rarity_code,offender_persistence_proxy_score,offender_persistence_proxy_code
0,THEFT OF IDENTITY,structured_fraud,simple,common,0,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",violent_high_risk,complex,common,2,medium
2,THEFT OF IDENTITY,structured_fraud,simple,common,0,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,opportunistic_property,simple,common,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),organized_complex,complex,common,4,high
5,THEFT OF IDENTITY,structured_fraud,simple,common,0,low
6,THEFT OF IDENTITY,structured_fraud,moderate,common,0,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,vulnerable_targeted,complex,common,2,medium
8,THEFT OF IDENTITY,structured_fraud,simple,common,0,low
9,THEFT OF IDENTITY,structured_fraud,simple,common,0,low


## 5.32 Define behavioral anomaly flag

We derive a behavioral anomaly indicator for potentially exceptional or unusual cases.

In [95]:
# [5.32] Define behavioral anomaly function

def derive_behavioral_anomaly_flag(
    behavior_signature_rarity_code,
    behavioral_threat_code,
    behavioral_diversity_code,
    escalation_potential_code,
    criminal_sophistication_code
):
    anomaly_flag = 0

    # Rare high threat behavior
    if (
        behavior_signature_rarity_code == "rare"
        and behavioral_threat_code == "high"
    ):
        anomaly_flag = 1

    # High diversity + high escalation
    elif (
        behavioral_diversity_code == "high"
        and escalation_potential_code in ["medium", "high"]
    ):
        anomaly_flag = 1

    # Vulnerable sophisticated cases
    elif (
        criminal_sophistication_code == "high"
        and behavioral_threat_code == "high"
    ):
        anomaly_flag = 1

    return anomaly_flag

## 5.33 Apply behavioral anomaly flag

We apply the behavioral anomaly rule to identify potentially exceptional behavior patterns.

In [96]:
# [5.33] Apply behavioral anomaly flag

df["behavioral_anomaly_flag"] = df.apply(
    lambda row: derive_behavioral_anomaly_flag(
        row["behavior_signature_rarity_code"],
        row["behavioral_threat_code"],
        row["behavioral_diversity_code"],
        row["escalation_potential_code"],
        row["criminal_sophistication_code"]
    ),
    axis=1
)

df[
    [
        "crime_description",
        "behavior_signature_rarity_code",
        "behavioral_threat_code",
        "behavioral_diversity_code",
        "escalation_potential_code",
        "behavioral_anomaly_flag"
    ]
].head(12)

,crime_description,behavior_signature_rarity_code,behavioral_threat_code,behavioral_diversity_code,escalation_potential_code,behavioral_anomaly_flag
0,THEFT OF IDENTITY,common,low,low,low,0
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",common,high,medium,high,1
2,THEFT OF IDENTITY,common,low,low,low,0
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,common,low,low,low,0
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),common,high,medium,low,0
5,THEFT OF IDENTITY,common,low,low,low,0
6,THEFT OF IDENTITY,common,low,low,low,0
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,common,high,medium,medium,0
8,THEFT OF IDENTITY,common,low,low,low,0
9,THEFT OF IDENTITY,common,low,low,low,0


## 6. Criminal Behavior Enrichment Milestone Summary

In this section we built a criminal behavior intelligence enrichment layer, including:

- crime family and severity taxonomy
- weapon and violence indicators
- modus operandi complexity features
- target context profiling
- composite behavioral risk profiles
- criminal sophistication scoring
- escalation and threat indicators
- offender persistence proxy
- behavioral diversity and anomaly detection
- behavioral fingerprint signatures and rarity indicators
- semantic behavior signature text for future embedding and LLM enrichment

Current outputs support:
- predictive modeling features
- behavioral intelligence analysis
- anomaly detection
- future semantic/graph-based offender pattern analysis

### Next Section

Next we can move into contextual enrichment (events, weather, calendar, social context) or formalize the CRIMEX schema and pipeline packaging.

## 6. Multilingual Crime Ontology Foundations

We add multilingual normalization foundations so CRIMEX can support crime data from multiple languages through canonical crime ontology mapping.

## 6.1 Detect crime description language

We define a lightweight language detection function for crime descriptions.

In [97]:
# [6.1] Define crime description language detection

import re

def detect_crime_language(crime_description):
    # Handle missing values
    if pd.isna(crime_description):
        return "unknown"

    text = str(crime_description)

    # Detect Arabic script
    if re.search(r'[\u0600-\u06FF]', text):
        return "ar"

    # Default English for current data
    return "en"

## 6.2 Apply language detection

We detect the language of crime descriptions and create language metadata.

In [98]:
# [6.2] Apply crime language detection

df["crime_description_language_code"] = df[
    "crime_description"
].apply(
    detect_crime_language
)

df[
    [
        "crime_description",
        "crime_description_language_code"
    ]
].head(10)

,crime_description,crime_description_language_code
0,THEFT OF IDENTITY,en
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",en
2,THEFT OF IDENTITY,en
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,en
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),en
5,THEFT OF IDENTITY,en
6,THEFT OF IDENTITY,en
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,en
8,THEFT OF IDENTITY,en
9,THEFT OF IDENTITY,en


## 6.3 Create raw crime description field

We preserve the original crime description before normalization.

In [99]:
# [6.3] Preserve original crime description

df["crime_description_raw"] = df["crime_description"]

df[
    [
        "crime_description_raw",
        "crime_description_language_code"
    ]
].head(10)

,crime_description_raw,crime_description_language_code
0,THEFT OF IDENTITY,en
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",en
2,THEFT OF IDENTITY,en
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,en
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),en
5,THEFT OF IDENTITY,en
6,THEFT OF IDENTITY,en
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,en
8,THEFT OF IDENTITY,en
9,THEFT OF IDENTITY,en


## 6.4 Define multilingual crime normalization

We define a starter function to normalize multilingual crime descriptions into canonical English text.

In [100]:
# [6.4] Define multilingual crime normalization function

def normalize_crime_description(
    crime_description,
    language_code
):
    # Handle missing values
    if pd.isna(crime_description):
        return None

    # Normalize text
    text = str(crime_description).lower().strip()

    # Starter Arabic keyword mapping
    arabic_mapping = {
        "سرقة": "theft",
        "احتيال": "fraud",
        "اعتداء": "assault",
        "قتل": "homicide",
        "تزوير": "forgery",
        "مخدرات": "narcotics",
        "سلاح": "weapon",
        "مركبة": "vehicle",
        "طفل": "child"
    }

    # Apply Arabic mapping if Arabic text
    if language_code == "ar":
        normalized_terms = []

        for arabic_word, english_term in arabic_mapping.items():
            if arabic_word in text:
                normalized_terms.append(english_term)

        if normalized_terms:
            return " ".join(normalized_terms)

    # English text is kept as normalized lowercase
    return text

## 6.5 Apply multilingual crime normalization

We create a canonical normalized crime description for multilingual pipeline support.

In [101]:
# [6.5] Apply multilingual crime normalization

df["crime_description_canonical_en"] = df.apply(
    lambda row: normalize_crime_description(
        row["crime_description_raw"],
        row["crime_description_language_code"]
    ),
    axis=1
)

df[
    [
        "crime_description_raw",
        "crime_description_language_code",
        "crime_description_canonical_en"
    ]
].head(10)

,crime_description_raw,crime_description_language_code,crime_description_canonical_en
0,THEFT OF IDENTITY,en,theft of identity
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",en,"assault with deadly weapon, aggravated assault"
2,THEFT OF IDENTITY,en,theft of identity
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,en,theft from motor vehicle - grand ($950.01 and ...
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),en,theft from motor vehicle - petty ($950 & under)
5,THEFT OF IDENTITY,en,theft of identity
6,THEFT OF IDENTITY,en,theft of identity
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,en,crm agnst chld (13 or under) (14-15 & susp 10 ...
8,THEFT OF IDENTITY,en,theft of identity
9,THEFT OF IDENTITY,en,theft of identity


## 6.6 Map canonical description to crime ontology

We map normalized crime descriptions into a reusable crime ontology code and description.

In [102]:
# [6.6] Define crime ontology mapping function

def map_crime_ontology(canonical_description):
    # Handle missing values
    if pd.isna(canonical_description):
        return "unknown", "unknown crime ontology"

    # Normalize text
    text = str(canonical_description).lower()

    # Fraud / identity
    if "identity" in text or "fraud" in text or "forgery" in text:
        return "fraud_identity", "fraud or identity crime"

    # Violent crime
    if "assault" in text or "battery" in text or "weapon" in text or "homicide" in text:
        return "violent", "violent crime"

    # Property crime
    if "theft" in text or "burglary" in text or "vehicle" in text or "robbery" in text:
        return "property", "property crime"

    # Child related
    if "child" in text or "chld" in text:
        return "child_related", "child related crime"

    # Narcotics
    if "narcotics" in text or "drug" in text:
        return "narcotics", "narcotics crime"

    # Default
    return "other", "other crime"

## 6.7 Apply crime ontology mapping

We apply ontology mapping to create language-independent crime family fields.

In [103]:
# [6.7] Apply crime ontology mapping

df[
    [
        "crime_ontology_code",
        "crime_ontology_desc"
    ]
] = df["crime_description_canonical_en"].apply(
    lambda x: pd.Series(
        map_crime_ontology(x)
    )
)

df[
    [
        "crime_description_raw",
        "crime_description_canonical_en",
        "crime_ontology_code",
        "crime_ontology_desc"
    ]
].head(10)

,crime_description_raw,crime_description_canonical_en,crime_ontology_code,crime_ontology_desc
0,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT","assault with deadly weapon, aggravated assault",violent,violent crime
2,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,theft from motor vehicle - grand ($950.01 and ...,property,property crime
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),theft from motor vehicle - petty ($950 & under),property,property crime
5,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime
6,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,crm agnst chld (13 or under) (14-15 & susp 10 ...,child_related,child related crime
8,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime
9,THEFT OF IDENTITY,theft of identity,fraud_identity,fraud or identity crime


## 6.8 Test Arabic ontology mapping

We test Arabic crime descriptions to confirm multilingual normalization works.

In [104]:
# [6.8] Test Arabic crime normalization and ontology mapping

arabic_test_df = pd.DataFrame({
    "crime_description_raw": [
        "سرقة مركبة",
        "احتيال مالي",
        "اعتداء بسلاح",
        "جريمة ضد طفل"
    ]
})

arabic_test_df["crime_description_language_code"] = arabic_test_df[
    "crime_description_raw"
].apply(
    detect_crime_language
)

arabic_test_df["crime_description_canonical_en"] = arabic_test_df.apply(
    lambda row: normalize_crime_description(
        row["crime_description_raw"],
        row["crime_description_language_code"]
    ),
    axis=1
)

arabic_test_df[
    [
        "crime_ontology_code",
        "crime_ontology_desc"
    ]
] = arabic_test_df["crime_description_canonical_en"].apply(
    lambda x: pd.Series(
        map_crime_ontology(x)
    )
)

arabic_test_df

,crime_description_raw,crime_description_language_code,crime_description_canonical_en,crime_ontology_code,crime_ontology_desc
0,سرقة مركبة,ar,theft vehicle,property,property crime
1,احتيال مالي,ar,fraud,fraud_identity,fraud or identity crime
2,اعتداء بسلاح,ar,assault weapon,violent,violent crime
3,جريمة ضد طفل,ar,child,child_related,child related crime


## 7. Multilingual Crime Ontology Milestone Summary

In this section we added multilingual crime normalization foundations, including:

- language detection for crime descriptions
- preservation of raw source crime text
- canonical normalized crime descriptions
- multilingual crime lexicon foundations
- crime ontology mapping independent of source language
- validation on Arabic and English crime descriptions

Current outputs support:

- multilingual source ingestion
- language-agnostic behavior enrichment
- future cross-lingual semantic crime modeling
- future multilingual LLM and embedding enrichment

### Next Section

Next we move into contextual enrichment, including environmental, temporal-social, event, and weather context features.

## 8. Global Contextual Enrichment

We build contextual features that depend on both time and location, including localized seasonality, daylight/darkness, country-specific calendars, and future weather/event enrichment.

## 8.1 Define hemisphere from latitude

We derive hemisphere because seasonality depends on geographic location.

In [105]:
# [8.1] Define hemisphere function

def get_hemisphere_code(latitude):
    # Handle missing latitude
    if pd.isna(latitude):
        return "unknown"

    # Northern hemisphere
    if latitude > 0:
        return "north"

    # Southern hemisphere
    if latitude < 0:
        return "south"

    # Equator
    return "equator"

## 8.2 Apply hemisphere feature

We derive hemisphere from final latitude to support localized seasonality.

In [106]:
# [8.2] Apply hemisphere feature

df["hemisphere_code"] = df["latitude_final"].apply(
    get_hemisphere_code
)

# Inspect result
df[
    [
        "latitude_final",
        "hemisphere_code"
    ]
].head(10)

,latitude_final,hemisphere_code
0,34.2124,north
1,34.1993,north
2,34.1847,north
3,34.0339,north
4,33.9813,north
5,34.0830,north
6,34.0100,north
7,34.1107,north
8,34.2763,north
9,34.1493,north


## 8.3 Define localized season

We derive season based on both month and hemisphere.

In [107]:
# [8.3] Define localized season function

def get_local_season_code(month, hemisphere_code):
    # Handle missing values
    if pd.isna(month) or hemisphere_code == "unknown":
        return "unknown"

    # Northern hemisphere seasons
    if hemisphere_code == "north":
        if month in [12, 1, 2]:
            return "winter"
        if month in [3, 4, 5]:
            return "spring"
        if month in [6, 7, 8]:
            return "summer"
        return "autumn"

    # Southern hemisphere seasons are reversed
    if hemisphere_code == "south":
        if month in [12, 1, 2]:
            return "summer"
        if month in [3, 4, 5]:
            return "autumn"
        if month in [6, 7, 8]:
            return "winter"
        return "spring"

    # Equator has less traditional seasonality
    return "tropical"

## 8.4 Apply localized season

We create localized season codes using month and hemisphere.

In [108]:
# [8.4] Apply localized season

df["local_season_code"] = df.apply(
    lambda row: get_local_season_code(
        row["month"],
        row["hemisphere_code"]
    ),
    axis=1
)

df[
    [
        "datetime_occured",
        "month",
        "hemisphere_code",
        "local_season_code"
    ]
].head(10)

,datetime_occured,month,hemisphere_code,local_season_code
0,2020-11-07 08:45:00,11,north,autumn
1,2020-10-18 18:45:00,10,north,autumn
2,2020-10-30 12:40:00,10,north,autumn
3,2020-12-24 13:10:00,12,north,winter
4,2020-09-29 18:30:00,9,north,autumn
5,2020-11-11 12:10:00,11,north,autumn
6,2020-04-16 13:50:00,4,north,spring
7,2020-07-07 14:00:00,7,north,summer
8,2020-03-02 12:00:00,3,north,spring
9,2020-09-01 09:00:00,9,north,autumn


## 8.5 Define daylight context

We derive basic daylight context as a first opportunity-context feature.

In [109]:
# [8.5] Define daylight context function

def derive_daylight_context(hour):
    # Handle missing
    if pd.isna(hour):
        return (
            "unknown",
            0,
            0
        )

    # Approximation for prototype
    # Later can replace with real sunrise/sunset APIs

    if 6 <= hour < 18:
        return (
            "daylight",
            1,
            0
        )

    return (
        "darkness",
        0,
        1
    )

## 8.6 Apply daylight context

We create daylight and darkness indicators from the incident hour.

In [110]:
# [8.6] Apply daylight context

df[
    [
        "daylight_context_code",
        "daylight_flag",
        "darkness_flag"
    ]
] = df["hour"].apply(
    lambda x: pd.Series(
        derive_daylight_context(x)
    )
)

df[
    [
        "datetime_occured",
        "hour",
        "daylight_context_code",
        "daylight_flag",
        "darkness_flag"
    ]
].head(10)

,datetime_occured,hour,daylight_context_code,daylight_flag,darkness_flag
0,2020-11-07 08:45:00,8,daylight,1,0
1,2020-10-18 18:45:00,18,darkness,0,1
2,2020-10-30 12:40:00,12,daylight,1,0
3,2020-12-24 13:10:00,13,daylight,1,0
4,2020-09-29 18:30:00,18,darkness,0,1
5,2020-11-11 12:10:00,12,daylight,1,0
6,2020-04-16 13:50:00,13,daylight,1,0
7,2020-07-07 14:00:00,14,daylight,1,0
8,2020-03-02 12:00:00,12,daylight,1,0
9,2020-09-01 09:00:00,9,daylight,1,0


## 8.7 Define opportunity context

We derive an opportunity context proxy using temporal-social conditions.

In [111]:
# [8.7] Define opportunity context function

def derive_opportunity_context(
    darkness_flag,
    is_weekend,
    target_context_code
):
    opportunity_score = 0

    # Darkness opportunity
    if darkness_flag == 1:
        opportunity_score += 1

    # Weekend activity opportunity
    if is_weekend == 1:
        opportunity_score += 1

    # Public space exposure
    if target_context_code == "public_space":
        opportunity_score += 1

    # Opportunity classification
    if opportunity_score <= 1:
        opportunity_code = "low"

    elif opportunity_score == 2:
        opportunity_code = "medium"

    else:
        opportunity_code = "high"

    return (
        opportunity_score,
        opportunity_code
    )

## 8.8 Apply opportunity context

We apply the opportunity context proxy using darkness, weekend, and target context.

In [112]:
# [8.8] Apply opportunity context

# Ensure weekend flag is numeric for modeling consistency
df["is_weekend"] = df["is_weekend"].astype(int)

df[
    [
        "opportunity_context_score",
        "opportunity_context_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_opportunity_context(
            row["darkness_flag"],
            row["is_weekend"],
            row["target_context_code"]
        )
    ),
    axis=1
)

df[
    [
        "datetime_occured",
        "darkness_flag",
        "is_weekend",
        "target_context_code",
        "opportunity_context_score",
        "opportunity_context_code"
    ]
].head(10)

,datetime_occured,darkness_flag,is_weekend,target_context_code,opportunity_context_score,opportunity_context_code
0,2020-11-07 08:45:00,0,1,identity,1,low
1,2020-10-18 18:45:00,1,1,public_space,3,high
2,2020-10-30 12:40:00,0,0,identity,0,low
3,2020-12-24 13:10:00,0,0,vehicle,0,low
4,2020-09-29 18:30:00,1,0,vehicle,1,low
5,2020-11-11 12:10:00,0,0,identity,0,low
6,2020-04-16 13:50:00,0,0,identity,0,low
7,2020-07-07 14:00:00,0,0,child,0,low
8,2020-03-02 12:00:00,0,0,identity,0,low
9,2020-09-01 09:00:00,0,0,identity,0,low


## 8.9 Define guardianship proxy

We derive a guardianship proxy based on opportunity and environmental protection indicators.

In [113]:
# [8.9] Define guardianship proxy function

def derive_guardianship_proxy(
    daylight_flag,
    target_context_code,
    darkness_flag
):
    guardianship_score = 0

    # Daylight increases informal guardianship
    if daylight_flag == 1:
        guardianship_score += 1

    # Residential areas may have higher guardianship
    if target_context_code == "residential":
        guardianship_score += 1

    # Darkness reduces guardianship
    if darkness_flag == 1:
        guardianship_score -= 1

    # Map guardianship level
    if guardianship_score <= 0:
        guardianship_code = "low"

    elif guardianship_score == 1:
        guardianship_code = "medium"

    else:
        guardianship_code = "high"

    return (
        guardianship_score,
        guardianship_code
    )

## 8.10 Apply guardianship proxy

We apply the guardianship proxy using daylight, darkness, and target context.

In [114]:
# [8.10] Apply guardianship proxy

df[
    [
        "guardianship_proxy_score",
        "guardianship_proxy_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_guardianship_proxy(
            row["daylight_flag"],
            row["target_context_code"],
            row["darkness_flag"]
        )
    ),
    axis=1
)

df[
    [
        "datetime_occured",
        "daylight_flag",
        "darkness_flag",
        "target_context_code",
        "guardianship_proxy_score",
        "guardianship_proxy_code"
    ]
].head(10)

,datetime_occured,daylight_flag,darkness_flag,target_context_code,guardianship_proxy_score,guardianship_proxy_code
0,2020-11-07 08:45:00,1,0,identity,1,medium
1,2020-10-18 18:45:00,0,1,public_space,-1,low
2,2020-10-30 12:40:00,1,0,identity,1,medium
3,2020-12-24 13:10:00,1,0,vehicle,1,medium
4,2020-09-29 18:30:00,0,1,vehicle,-1,low
5,2020-11-11 12:10:00,1,0,identity,1,medium
6,2020-04-16 13:50:00,1,0,identity,1,medium
7,2020-07-07 14:00:00,1,0,child,1,medium
8,2020-03-02 12:00:00,1,0,identity,1,medium
9,2020-09-01 09:00:00,1,0,identity,1,medium


## 8.11 Define holiday context intensity

We derive holiday opportunity context using holiday proximity and weekend conditions.

In [115]:
# [8.11] Define holiday opportunity context function

def derive_holiday_context(
    is_holiday,
    is_weekend,
    days_to_holiday,
    days_from_holiday
):
    holiday_context_score = 0

    # Holiday day itself
    if is_holiday == 1:
        holiday_context_score += 2

    # Weekend social activity
    if is_weekend == 1:
        holiday_context_score += 1

    # Near upcoming holiday
    if (
        pd.notna(days_to_holiday)
        and days_to_holiday <= 2
        and days_to_holiday >= 0
    ):
        holiday_context_score += 1

    # Just after holiday
    if (
        pd.notna(days_from_holiday)
        and days_from_holiday <= 2
        and days_from_holiday >= 0
    ):
        holiday_context_score += 1

    # Context level
    if holiday_context_score <= 1:
        holiday_context_code = "low"

    elif holiday_context_score <= 2:
        holiday_context_code = "medium"

    else:
        holiday_context_code = "high"

    return (
        holiday_context_score,
        holiday_context_code
    )

## 8.12 Check holiday columns availability

We check whether holiday-related columns already exist before applying holiday context.

In [116]:
# [8.12] Check holiday-related columns

holiday_columns = [
    "is_holiday",
    "days_to_holiday",
    "days_from_holiday"
]

available_holiday_columns = [
    col for col in holiday_columns
    if col in df.columns
]

missing_holiday_columns = [
    col for col in holiday_columns
    if col not in df.columns
]

print("Available holiday columns:", available_holiday_columns)
print("Missing holiday columns   :", missing_holiday_columns)

Available holiday columns: []
Missing holiday columns   : ['is_holiday', 'days_to_holiday', 'days_from_holiday']


## 8.13 Define holiday calendar source

We define the first country holiday calendar used for crime-location contextual enrichment.

In [117]:
# [8.13] Define holiday calendar source

HOLIDAY_CONFIG = {
    "holiday_country_code": "US",
    "holiday_source_code": "manual_us_federal_v1",
    "holiday_calendar_scope_desc": "United States federal holidays"
}

HOLIDAY_CONFIG

{'holiday_country_code': 'US',
 'holiday_source_code': 'manual_us_federal_v1',
 'holiday_calendar_scope_desc': 'United States federal holidays'}

## 8.14 Create US holiday calendar

We create a reusable holiday calendar dataframe for the dataset years.

In [118]:
# [8.14] Create US holiday calendar for dataset years

from pandas.tseries.holiday import USFederalHolidayCalendar

def create_us_holiday_calendar(start_date, end_date):
    # Create US federal holiday calendar
    calendar = USFederalHolidayCalendar()

    # Generate holidays in date range
    holiday_dates = calendar.holidays(
        start=start_date,
        end=end_date
    )

    # Convert holidays into dataframe
    holiday_df = pd.DataFrame({
        "holiday_date": holiday_dates
    })

    # Add calendar metadata
    holiday_df["holiday_country_code"] = HOLIDAY_CONFIG["holiday_country_code"]
    holiday_df["holiday_source_code"] = HOLIDAY_CONFIG["holiday_source_code"]

    return holiday_df

## 8.15 Build holiday calendar for dataset range

We generate holiday dates using the minimum and maximum crime occurrence dates.

In [119]:
# [8.15] Build holiday calendar for dataset date range

holiday_df = create_us_holiday_calendar(
    start_date=df["date_occured"].min(),
    end_date=df["date_occured"].max()
)

print("Holiday rows:", len(holiday_df))

holiday_df.head(10)

Holiday rows: 54


,holiday_date,holiday_country_code,holiday_source_code
0,2020-01-01,US,manual_us_federal_v1
1,2020-01-20,US,manual_us_federal_v1
2,2020-02-17,US,manual_us_federal_v1
3,2020-05-25,US,manual_us_federal_v1
4,2020-07-03,US,manual_us_federal_v1
5,2020-09-07,US,manual_us_federal_v1
6,2020-10-12,US,manual_us_federal_v1
7,2020-11-11,US,manual_us_federal_v1
8,2020-11-26,US,manual_us_federal_v1
9,2020-12-25,US,manual_us_federal_v1


## 8.16 Create holiday indicators

We derive holiday indicators and distance-to-holiday features.

In [120]:
# [8.16] Create holiday indicator features

# Normalize to dates
df["date_occured_only"] = pd.to_datetime(
    df["date_occured"]
).dt.normalize()

holiday_set = set(
    holiday_df["holiday_date"]
)

# Holiday flag
df["is_holiday"] = df[
    "date_occured_only"
].isin(
    holiday_set
).astype(int)


def closest_holiday_features(crime_date):
    # Compute distances to all holidays
    day_diffs = [
        (crime_date - h).days
        for h in holiday_set
    ]

    # Closest previous holiday
    past_diffs = [
        d for d in day_diffs
        if d >= 0
    ]

    # Closest upcoming holiday
    future_diffs = [
        -d for d in day_diffs
        if d < 0
    ]

    days_from_holiday = (
        min(past_diffs)
        if len(past_diffs) > 0
        else None
    )

    days_to_holiday = (
        min(future_diffs)
        if len(future_diffs) > 0
        else None
    )

    return (
        days_to_holiday,
        days_from_holiday
    )


df[
    [
        "days_to_holiday",
        "days_from_holiday"
    ]
] = df["date_occured_only"].apply(
    lambda x: pd.Series(
        closest_holiday_features(x)
    )
)

## 8.17 Preview holiday features

We inspect the generated holiday indicators and proximity features.

In [121]:
# [8.17] Preview holiday features

df[
    [
        "date_occured_only",
        "is_holiday",
        "days_to_holiday",
        "days_from_holiday"
    ]
].head(10)

,date_occured_only,is_holiday,days_to_holiday,days_from_holiday
0,2020-11-07,0,4.0,26.0
1,2020-10-18,0,24.0,6.0
2,2020-10-30,0,12.0,18.0
3,2020-12-24,0,1.0,28.0
4,2020-09-29,0,13.0,22.0
5,2020-11-11,1,15.0,0.0
6,2020-04-16,0,39.0,59.0
7,2020-07-07,0,62.0,4.0
8,2020-03-02,0,84.0,14.0
9,2020-09-01,0,6.0,60.0


## 8.18 Apply holiday opportunity context

We derive holiday opportunity context using holiday activity and holiday proximity.

In [122]:
# [8.18] Apply holiday opportunity context

df[
    [
        "holiday_context_score",
        "holiday_context_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_holiday_context(
            row["is_holiday"],
            row["is_weekend"],
            row["days_to_holiday"],
            row["days_from_holiday"]
        )
    ),
    axis=1
)

df[
    [
        "date_occured_only",
        "is_holiday",
        "is_weekend",
        "days_to_holiday",
        "days_from_holiday",
        "holiday_context_score",
        "holiday_context_code"
    ]
].head(12)

,date_occured_only,is_holiday,is_weekend,days_to_holiday,days_from_holiday,holiday_context_score,holiday_context_code
0,2020-11-07,0,1,4.0,26.0,1,low
1,2020-10-18,0,1,24.0,6.0,1,low
2,2020-10-30,0,0,12.0,18.0,0,low
3,2020-12-24,0,0,1.0,28.0,1,low
4,2020-09-29,0,0,13.0,22.0,0,low
5,2020-11-11,1,0,15.0,0.0,3,high
6,2020-04-16,0,0,39.0,59.0,0,low
7,2020-07-07,0,0,62.0,4.0,0,low
8,2020-03-02,0,0,84.0,14.0,0,low
9,2020-09-01,0,0,6.0,60.0,0,low


## 8.19 Define strain and stress context proxy

We derive a contextual strain proxy reflecting elevated situational pressure conditions.

In [123]:
# [8.19] Define strain context proxy function

def derive_strain_context(
    opportunity_context_score,
    holiday_context_score,
    darkness_flag,
    behavioral_threat_score
):
    strain_score = 0

    # Opportunity exposure contribution
    if opportunity_context_score >= 2:
        strain_score += 1

    # Holiday social pressure contribution
    if holiday_context_score >= 2:
        strain_score += 1

    # Darkness contribution
    if darkness_flag == 1:
        strain_score += 1

    # High threat behavior contribution
    if behavioral_threat_score >= 8:
        strain_score += 1

    # Context level
    if strain_score <= 1:
        strain_code = "low"

    elif strain_score <= 2:
        strain_code = "medium"

    else:
        strain_code = "high"

    return (
        strain_score,
        strain_code
    )

## 8.20 Apply strain and stress context proxy

We apply the contextual strain proxy using opportunity, holiday, darkness, and behavioral threat conditions.

In [124]:
# [8.20] Apply strain/stress context proxy

df[
    [
        "strain_context_score",
        "strain_context_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_strain_context(
            row["opportunity_context_score"],
            row["holiday_context_score"],
            row["darkness_flag"],
            row["behavioral_threat_score"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "opportunity_context_score",
        "holiday_context_score",
        "behavioral_threat_score",
        "strain_context_score",
        "strain_context_code"
    ]
].head(12)

,crime_description,opportunity_context_score,holiday_context_score,behavioral_threat_score,strain_context_score,strain_context_code
0,THEFT OF IDENTITY,1,1,2,0,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",3,1,20,3,high
2,THEFT OF IDENTITY,0,0,2,0,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,0,1,0,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1,0,9,2,medium
5,THEFT OF IDENTITY,0,3,2,1,low
6,THEFT OF IDENTITY,0,0,4,0,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,0,0,16,1,low
8,THEFT OF IDENTITY,0,0,2,0,low
9,THEFT OF IDENTITY,0,0,2,0,low


## 8.21 Define routine activity risk proxy

We derive a routine activity risk proxy based on opportunity and guardianship conditions.

In [125]:
# [8.21] Define routine activity risk proxy

def derive_routine_activity_risk(
    opportunity_context_score,
    guardianship_proxy_code,
    darkness_flag
):
    risk_score = 0

    # Opportunity contribution
    if opportunity_context_score >= 2:
        risk_score += 1

    # Low guardianship contribution
    if guardianship_proxy_code == "low":
        risk_score += 1

    # Darkness contribution
    if darkness_flag == 1:
        risk_score += 1

    # Risk level
    if risk_score <= 1:
        risk_code = "low"

    elif risk_score == 2:
        risk_code = "medium"

    else:
        risk_code = "high"

    return (
        risk_score,
        risk_code
    )

## 8.22 Apply routine activity risk proxy

We apply a routine activity theory risk proxy using opportunity and guardianship conditions.

In [126]:
# [8.22] Apply routine activity risk proxy

df[
    [
        "routine_activity_risk_score",
        "routine_activity_risk_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_routine_activity_risk(
            row["opportunity_context_score"],
            row["guardianship_proxy_code"],
            row["darkness_flag"]
        )
    ),
    axis=1
)

df[
    [
        "crime_description",
        "opportunity_context_score",
        "guardianship_proxy_code",
        "darkness_flag",
        "routine_activity_risk_score",
        "routine_activity_risk_code"
    ]
].head(12)

,crime_description,opportunity_context_score,guardianship_proxy_code,darkness_flag,routine_activity_risk_score,routine_activity_risk_code
0,THEFT OF IDENTITY,1,medium,0,0,low
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",3,low,1,3,high
2,THEFT OF IDENTITY,0,medium,0,0,low
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,0,medium,0,0,low
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1,low,1,2,medium
5,THEFT OF IDENTITY,0,medium,0,0,low
6,THEFT OF IDENTITY,0,medium,0,0,low
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,0,medium,0,0,low
8,THEFT OF IDENTITY,0,medium,0,0,low
9,THEFT OF IDENTITY,0,medium,0,0,low


## 8.23 Define weather stress proxy

We define a prototype weather stress proxy for later replacement with real historical weather enrichment.

In [127]:
# [8.23] Define weather stress proxy function

def derive_weather_stress_proxy(
    local_season_code,
    month
):
    weather_stress_score = 0

    # Prototype seasonal stress assumptions
    # Later replace with actual weather data

    # Summer stress
    if local_season_code == "summer":
        weather_stress_score += 1

    # Peak summer months stronger stress
    if month in [7, 8]:
        weather_stress_score += 1

    # Risk code
    if weather_stress_score == 0:
        weather_stress_code = "low"

    elif weather_stress_score == 1:
        weather_stress_code = "medium"

    else:
        weather_stress_code = "high"

    return (
        weather_stress_score,
        weather_stress_code
    )

## 8.24 Apply weather stress proxy

We apply the prototype weather stress proxy using localized seasonality.

In [128]:
# [8.24] Apply weather stress proxy

df[
    [
        "weather_stress_score",
        "weather_stress_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_weather_stress_proxy(
            row["local_season_code"],
            row["month"]
        )
    ),
    axis=1
)

df[
    [
        "datetime_occured",
        "local_season_code",
        "month",
        "weather_stress_score",
        "weather_stress_code"
    ]
].head(12)

,datetime_occured,local_season_code,month,weather_stress_score,weather_stress_code
0,2020-11-07 08:45:00,autumn,11,0,low
1,2020-10-18 18:45:00,autumn,10,0,low
2,2020-10-30 12:40:00,autumn,10,0,low
3,2020-12-24 13:10:00,winter,12,0,low
4,2020-09-29 18:30:00,autumn,9,0,low
5,2020-11-11 12:10:00,autumn,11,0,low
6,2020-04-16 13:50:00,spring,4,0,low
7,2020-07-07 14:00:00,summer,7,2,high
8,2020-03-02 12:00:00,spring,3,0,low
9,2020-09-01 09:00:00,autumn,9,0,low


In [129]:
df.shape

(1004894, 105)

## 8.25 Define weather disruption flag

We derive a prototype environmental disruption flag related to extreme weather conditions.

In [130]:
# [8.25] Define weather disruption flag function

def derive_weather_disruption_flag(
    weather_stress_code
):
    # Prototype logic
    # Later replaced with real severe weather indicators

    if weather_stress_code == "high":
        return 1

    return 0

## 8.26 Apply weather disruption flag

We apply a prototype environmental disruption indicator.

In [131]:
# [8.26] Apply weather disruption flag

df["weather_disruption_flag"] = df[
    "weather_stress_code"
].apply(
    derive_weather_disruption_flag
)

df[
    [
        "datetime_occured",
        "weather_stress_code",
        "weather_disruption_flag"
    ]
].head(12)

,datetime_occured,weather_stress_code,weather_disruption_flag
0,2020-11-07 08:45:00,low,0
1,2020-10-18 18:45:00,low,0
2,2020-10-30 12:40:00,low,0
3,2020-12-24 13:10:00,low,0
4,2020-09-29 18:30:00,low,0
5,2020-11-11 12:10:00,low,0
6,2020-04-16 13:50:00,low,0
7,2020-07-07 14:00:00,high,1
8,2020-03-02 12:00:00,low,0
9,2020-09-01 09:00:00,low,0


## 9. Contextual Enrichment Milestone Summary

In this section we built a geo-temporal contextual intelligence layer including:

### Localized environmental context
- hemisphere-aware seasonality
- localized season codes
- daylight and darkness context
- prototype weather stress and disruption indicators

### Opportunity and guardianship context
- opportunity context proxy
- guardianship proxy
- routine activity risk proxy

### Social and event context
- holiday indicators
- holiday proximity features
- holiday opportunity context

### Advanced criminological context
- strain and stress context proxy

Current outputs support:
- crime opportunity modeling
- environmental and social condition analysis
- criminology theory-inspired feature engineering
- future weather/event enrichment plug-ins

### CRIMEX core pillars built so far
- temporal enrichment
- geo enrichment
- behavioral intelligence enrichment
- contextual enrichment

Next phases can include:
- graph/network offender relationships
- advanced historical weather APIs
- socioeconomic neighborhood context
- final pipeline packaging and reusable automation scripts

## 10. CRIMEX Pipeline Packaging Design

We begin defining the reusable CRIMEX ingestion and enrichment pipeline architecture for scalable processing of new criminal data from heterogeneous sources.

Pipeline requirements:

- support minimal or incomplete source records
- support multilingual source data
- support country-aware geo-temporal enrichment
- preserve source provenance metadata
- support schema evolution as enrichment features grow
- support incremental merging of newly enriched records with prior CRIMEX master data

Target:
Notebook logic later becomes reusable Python pipeline modules.

## 10.1 Define minimum input schema

We define the minimum fields required for CRIMEX ingestion, including fallback handling for missing values.

In [132]:
# [10.1] Define minimum CRIMEX input schema

CRIMEX_MIN_INPUT_SCHEMA = {
    "case_id":
        "unique incident identifier",

    "crime_description":
        "raw offense description in any language",

    "datetime_occured":
        "incident datetime if available",

    "latitude":
        "optional latitude",

    "longitude":
        "optional longitude",

    "country_code":
        "optional source country code",

    "source_system_code":
        "source of collected data",

    "source_record_url":
        "optional source record url",

    "source_language_code":
        "optional if known"
}

CRIMEX_MIN_INPUT_SCHEMA

{'case_id': 'unique incident identifier',
 'crime_description': 'raw offense description in any language',
 'datetime_occured': 'incident datetime if available',
 'latitude': 'optional latitude',
 'longitude': 'optional longitude',
 'country_code': 'optional source country code',
 'source_system_code': 'source of collected data',
 'source_record_url': 'optional source record url',
 'source_language_code': 'optional if known'}

## 10.2 Define canonical output schema groups

We define the main output schema groups so new enriched fields can be organized clearly.

In [133]:
# [10.2] Define canonical CRIMEX output schema groups

CRIMEX_OUTPUT_SCHEMA_GROUPS = {
    "source_provenance": [
        "source_system_code",
        "source_record_url",
        "source_language_code"
    ],

    "core_incident": [
        "case_id",
        "crime_description_raw",
        "crime_description_canonical_en",
        "datetime_occured"
    ],

    "geo_features": [
        "latitude_final",
        "longitude_final",
        "city_desc",
        "state_desc",
        "country_code"
    ],

    "temporal_features": [
        "year",
        "month",
        "day_of_week_number",
        "hour",
        "local_season_code"
    ],

    "behavior_features": [
        "crime_ontology_code",
        "behavior_risk_profile_code",
        "criminal_sophistication_score",
        "behavioral_threat_score"
    ],

    "context_features": [
        "opportunity_context_score",
        "guardianship_proxy_score",
        "holiday_context_score",
        "weather_stress_score"
    ]
}

CRIMEX_OUTPUT_SCHEMA_GROUPS

{'source_provenance': ['source_system_code',
  'source_record_url',
  'source_language_code'],
 'core_incident': ['case_id',
  'crime_description_raw',
  'crime_description_canonical_en',
  'datetime_occured'],
 'geo_features': ['latitude_final',
  'longitude_final',
  'city_desc',
  'state_desc',
  'country_code'],
 'temporal_features': ['year',
  'month',
  'day_of_week_number',
  'hour',
  'local_season_code'],
 'behavior_features': ['crime_ontology_code',
  'behavior_risk_profile_code',
  'criminal_sophistication_score',
  'behavioral_threat_score'],
 'context_features': ['opportunity_context_score',
  'guardianship_proxy_score',
  'holiday_context_score',
  'weather_stress_score']}

## 10.3 Define actor intelligence schema group

We add an actor intelligence schema layer for offender-related attributes when available from source data.

In [134]:
# [10.3] Define actor intelligence schema group

CRIMEX_OUTPUT_SCHEMA_GROUPS["actor_features"] = [
    "offender_age",
    "offender_age_group",
    "offender_gender_code",
    "offender_nationality_code",

    "repeat_offender_flag",
    "prior_offense_count",
    "offense_specialization_score",

    "solo_vs_group_code",
    "organized_group_flag",

    "cross_border_actor_flag"
]

CRIMEX_OUTPUT_SCHEMA_GROUPS["actor_features"]

['offender_age',
 'offender_age_group',
 'offender_gender_code',
 'offender_nationality_code',
 'repeat_offender_flag',
 'prior_offense_count',
 'offense_specialization_score',
 'solo_vs_group_code',
 'organized_group_flag',
 'cross_border_actor_flag']

## 10.4 Expand source provenance metadata

We expand source provenance fields to track origin, reliability, and validation of collected data.

In [135]:
# [10.4] Expand source provenance schema

CRIMEX_OUTPUT_SCHEMA_GROUPS[
    "source_provenance"
] += [
    "source_reliability_score",
    "source_collection_method_code",
    "record_validation_status",
    "record_ingestion_timestamp",
    "enrichment_pipeline_version"
]

CRIMEX_OUTPUT_SCHEMA_GROUPS[
    "source_provenance"
]

['source_system_code',
 'source_record_url',
 'source_language_code',
 'source_reliability_score',
 'source_collection_method_code',
 'record_validation_status',
 'record_ingestion_timestamp',
 'enrichment_pipeline_version']

## 10.5 Define sparse input fallback rules

We define fallback rules so CRIMEX can ingest incomplete source records while still applying enrichment.

In [136]:
# [10.5] Define sparse input fallback rules

CRIMEX_FALLBACK_RULES = {

    "crime_description_missing":
        "route_to_unknown_offense_bucket",

    "datetime_missing":
        "allow_partial_record_without_temporal_enrichment",

    "coordinates_missing":
        "skip_geo_enrichment_use_country_if_available",

    "country_missing":
        "infer_from_coordinates_if_available",

    "language_missing":
        "auto_detect_language",

    "actor_fields_missing":
        "apply_null_safe_actor_enrichment"
}

CRIMEX_FALLBACK_RULES

{'crime_description_missing': 'route_to_unknown_offense_bucket',
 'datetime_missing': 'allow_partial_record_without_temporal_enrichment',
 'coordinates_missing': 'skip_geo_enrichment_use_country_if_available',
 'country_missing': 'infer_from_coordinates_if_available',
 'language_missing': 'auto_detect_language',
 'actor_fields_missing': 'apply_null_safe_actor_enrichment'}

## 10.6 Define source adapters

We define source adapter types for heterogeneous criminal data collection sources.

In [137]:
# [10.6] Define CRIMEX source adapters

CRIMEX_SOURCE_ADAPTERS = {

    "open_data_adapter":
        "government or public crime datasets",

    "court_records_adapter":
        "court and judicial records",

    "watchlist_adapter":
        "trusted sanctions or watchlist sources",

    "news_osint_adapter":
        "validated open source intelligence feeds",

    "manual_case_upload_adapter":
        "manual csv or analyst supplied records"
}

CRIMEX_SOURCE_ADAPTERS

{'open_data_adapter': 'government or public crime datasets',
 'court_records_adapter': 'court and judicial records',
 'watchlist_adapter': 'trusted sanctions or watchlist sources',
 'news_osint_adapter': 'validated open source intelligence feeds',
 'manual_case_upload_adapter': 'manual csv or analyst supplied records'}

## 10.7 Define incremental merge strategy

We define how newly enriched records will be merged with an existing CRIMEX master dataset.

In [138]:
# [10.7] Define incremental merge strategy

CRIMEX_MERGE_STRATEGY = {
    "primary_merge_key": "case_id",
    "secondary_merge_keys": [
        "source_system_code",
        "source_record_url"
    ],
    "duplicate_handling_code": "keep_highest_reliability_record",
    "schema_evolution_code": "allow_new_columns_with_null_backfill",
    "versioning_strategy_code": "create_new_dataset_version_per_merge"
}

CRIMEX_MERGE_STRATEGY

{'primary_merge_key': 'case_id',
 'secondary_merge_keys': ['source_system_code', 'source_record_url'],
 'duplicate_handling_code': 'keep_highest_reliability_record',
 'schema_evolution_code': 'allow_new_columns_with_null_backfill',
 'versioning_strategy_code': 'create_new_dataset_version_per_merge'}

## 10.8 Define CRIMEX pipeline execution order

We define the planned execution order for the reusable CRIMEX pipeline.

In [139]:
# [10.8] Define CRIMEX pipeline execution order

CRIMEX_PIPELINE_STEPS = [
    "01_source_ingestion",
    "02_schema_standardization",
    "03_language_detection",
    "04_multilingual_crime_ontology_mapping",
    "05_datetime_and_coordinate_validation",
    "06_temporal_feature_enrichment",
    "07_geo_and_poi_enrichment",
    "08_behavioral_intelligence_enrichment",
    "09_contextual_enrichment",
    "10_actor_intelligence_enrichment",
    "11_quality_validation",
    "12_incremental_merge",
    "13_dataset_version_export"
]

CRIMEX_PIPELINE_STEPS

['01_source_ingestion',
 '02_schema_standardization',
 '03_language_detection',
 '04_multilingual_crime_ontology_mapping',
 '05_datetime_and_coordinate_validation',
 '06_temporal_feature_enrichment',
 '07_geo_and_poi_enrichment',
 '08_behavioral_intelligence_enrichment',
 '09_contextual_enrichment',
 '10_actor_intelligence_enrichment',
 '11_quality_validation',
 '12_incremental_merge',
 '13_dataset_version_export']

## 10.9 Define pipeline backend options

We define switchable backend options for enrichment services used by the pipeline.

In [140]:
# [10.9] Define pipeline backend options

CRIMEX_BACKEND_OPTIONS = {
    "geo_backend_options": [
        "online",
        "local"
    ],

    "translation_backend_options": [
        "none",
        "rule_based_ontology",
        "ollama_local",
        "api_llm"
    ],

    "weather_backend_options": [
        "none",
        "prototype_seasonal_proxy",
        "historical_weather_api",
        "local_weather_dataset"
    ],

    "holiday_backend_options": [
        "manual_country_calendar",
        "python_calendar_library",
        "external_holiday_api"
    ]
}

CRIMEX_BACKEND_OPTIONS

{'geo_backend_options': ['online', 'local'],
 'translation_backend_options': ['none',
  'rule_based_ontology',
  'ollama_local',
  'api_llm'],
 'weather_backend_options': ['none',
  'prototype_seasonal_proxy',
  'historical_weather_api',
  'local_weather_dataset'],
 'holiday_backend_options': ['manual_country_calendar',
  'python_calendar_library',
  'external_holiday_api']}

## 10.10 Define feature registry versioning

We define a registry structure for managing enrichment features added over time.

In [141]:
# [10.10] Define enrichment feature registry

CRIMEX_FEATURE_REGISTRY = {
    "temporal_features_version":
        "v1",

    "geo_features_version":
        "v1",

    "behavior_features_version":
        "v1",

    "context_features_version":
        "v1",

    "actor_features_version":
        "v1",

    "schema_change_policy":
        "append_new_features_do_not_break_old_schema"
}

CRIMEX_FEATURE_REGISTRY

{'temporal_features_version': 'v1',
 'geo_features_version': 'v1',
 'behavior_features_version': 'v1',
 'context_features_version': 'v1',
 'actor_features_version': 'v1',
 'schema_change_policy': 'append_new_features_do_not_break_old_schema'}

## 10.11 Define main CRIMEX pipeline skeleton

We define the high-level reusable pipeline function that will later be moved into Python scripts.

In [142]:
# [10.11] Define main CRIMEX pipeline skeleton

def run_crimex_pipeline(input_df, pipeline_config):
    # Copy input data to avoid changing original dataframe
    df_pipeline = input_df.copy()

    # Step 1: standardize source schema
    # TODO: add source adapter logic

    # Step 2: detect language and normalize crime text
    # TODO: add multilingual normalization module

    # Step 3: validate datetime and coordinates
    # TODO: add datetime and coordinate validation modules

    # Step 4: add temporal features
    # TODO: add temporal feature module

    # Step 5: add geo and POI features
    # TODO: add geo enrichment module

    # Step 6: add behavioral intelligence features
    # TODO: add behavior enrichment module

    # Step 7: add contextual features
    # TODO: add context enrichment module

    # Step 8: add actor intelligence features
    # TODO: add actor enrichment module

    # Step 9: validate final output
    # TODO: add quality validation module

    return df_pipeline

## 10.12 Define incremental master update skeleton

We define the merge/update function used to combine new enriched data with an existing CRIMEX master dataset.

In [143]:
# [10.12] Define incremental master update skeleton

def merge_into_crimex_master(
    existing_master_df,
    new_enriched_df
):
    # Placeholder logic
    # Later:
    # 1 deduplicate by merge keys
    # 2 resolve conflicts by source reliability
    # 3 preserve schema evolution
    # 4 create new versioned master output

    merged_df = (
        pd.concat(
            [
                existing_master_df,
                new_enriched_df
            ],
            ignore_index=True
        )
        .drop_duplicates(
            subset=["case_id"],
            keep="last"
        )
    )

    return merged_df

## 10.13 Pipeline Packaging Milestone Summary

In this section we defined the reusable CRIMEX pipeline architecture, including:

- minimum input schema for sparse internet-collected records
- canonical output schema groups
- actor intelligence schema layer
- source provenance and reliability metadata
- fallback rules for missing fields
- source adapter types
- incremental merge strategy
- pipeline execution order
- switchable backend options
- feature registry versioning
- main pipeline function skeleton
- master dataset merge skeleton

This prepares CRIMEX to support:
- new data ingestion
- multilingual records
- country-aware enrichment
- local/online enrichment backends
- schema evolution
- incremental dataset versioning
- future Python script automation

Next phases can focus on:
- moving stable notebook logic into Python files
- creating command-line scripts
- adding actor intelligence enrichment
- adding advanced external data sources

## 10.14 Define pipeline feature toggles

We define configurable feature toggles and backend routing options for the CRIMEX pipeline.

In [144]:
# [10.14] Define pipeline feature toggles

CRIMEX_PIPELINE_CONFIG = {

    "enable_geo_enrichment": True,

    "geo_backend_code":
        "online",  # online or local

    "enable_poi_enrichment": True,

    "enable_translation_module": True,

    "translation_backend_code":
        "rule_based_ontology",

    "enable_weather_enrichment": False,

    "weather_backend_code":
        "prototype_seasonal_proxy",

    "enable_graph_features": True,
    "enable_incremental_merge": True,
    "enable_quality_validation": True,
    "fail_on_missing_required_fields": False,

    "enable_actor_enrichment": True
}

CRIMEX_PIPELINE_CONFIG

{'enable_geo_enrichment': True,
 'geo_backend_code': 'online',
 'enable_poi_enrichment': True,
 'enable_translation_module': True,
 'translation_backend_code': 'rule_based_ontology',
 'enable_weather_enrichment': False,
 'weather_backend_code': 'prototype_seasonal_proxy',
 'enable_graph_features': True,
 'enable_incremental_merge': True,
 'enable_quality_validation': True,
 'fail_on_missing_required_fields': False,
 'enable_actor_enrichment': True}

## 10.15 Define backend failover strategy

We define failover routing for enrichment backends.

In [145]:
# [10.15] Define backend failover strategy

CRIMEX_BACKEND_FAILOVER = {

    "geo_backend_failover": [
        "online",
        "local"
    ],

    "translation_failover": [
        "rule_based_ontology",
        "ollama_local"
    ],

    "weather_failover": [
        "historical_weather_api",
        "prototype_seasonal_proxy"
    ]
}

CRIMEX_BACKEND_FAILOVER

{'geo_backend_failover': ['online', 'local'],
 'translation_failover': ['rule_based_ontology', 'ollama_local'],
 'weather_failover': ['historical_weather_api', 'prototype_seasonal_proxy']}

## 11. Data Quality and Validation Layer

We begin defining quality controls to validate source records and enriched outputs before they enter the CRIMEX master dataset.

## 11.1 Define quality validation dimensions

We define the main dimensions used to assess source and enrichment quality.

In [146]:
# [11.1] Define quality validation dimensions

CRIMEX_QUALITY_DIMENSIONS = {
    "completeness":
        "required fields availability",

    "consistency":
        "logical consistency across fields",

    "validity":
        "values conform to expected rules",

    "provenance_trust":
        "source reliability confidence",

    "enrichment_quality":
        "confidence in generated features"
}

CRIMEX_QUALITY_DIMENSIONS

{'completeness': 'required fields availability',
 'consistency': 'logical consistency across fields',
 'validity': 'values conform to expected rules',
 'provenance_trust': 'source reliability confidence',
 'enrichment_quality': 'confidence in generated features'}

## 11.2 Define record quality score framework

We define a composite record quality scoring framework.

In [147]:
# [11.2] Define record quality scoring framework

def calculate_record_quality_score(
    completeness_score,
    consistency_score,
    validity_score,
    provenance_score,
    enrichment_score
):
    quality_score = (
        completeness_score
        + consistency_score
        + validity_score
        + provenance_score
        + enrichment_score
    )

    # Average score
    quality_score = quality_score / 5

    # Quality band
    if quality_score >= 0.80:
        quality_code = "high"

    elif quality_score >= 0.60:
        quality_code = "medium"

    else:
        quality_code = "low"

    return (
        quality_score,
        quality_code
    )

## 11.3 Define completeness score

We calculate record completeness based on availability of key CRIMEX fields.

In [148]:
# [11.3] Define completeness scoring function

def calculate_completeness_score(row, required_fields):
    # Count available required fields
    available_count = 0

    for field in required_fields:
        if field in row.index and pd.notna(row[field]):
            available_count += 1

    # Avoid division by zero
    if len(required_fields) == 0:
        return 0

    # Return completeness ratio
    return available_count / len(required_fields)

## 11.4 Define required quality fields

We define the key fields used to evaluate record completeness.

In [149]:
# [11.4] Define required completeness fields

QUALITY_REQUIRED_FIELDS = [
    "case_id",
    "crime_description_raw",
    "datetime_occured",
    "latitude_final",
    "longitude_final",
    "crime_ontology_code"
]

QUALITY_REQUIRED_FIELDS

['case_id',
 'crime_description_raw',
 'datetime_occured',
 'latitude_final',
 'longitude_final',
 'crime_ontology_code']

## 11.5 Apply completeness scoring

We calculate a completeness score for each record.

In [150]:
# [11.5] Apply completeness scoring

df["completeness_score"] = df.apply(
    lambda row: calculate_completeness_score(
        row,
        QUALITY_REQUIRED_FIELDS
    ),
    axis=1
)

df[
    [
        "case_id",
        "completeness_score"
    ]
].head(10)

,case_id,completeness_score
0,211507896,1.0
1,201516622,1.0
2,240913563,1.0
3,210704711,1.0
4,201418201,1.0
5,240412063,1.0
6,240317069,1.0
7,201115217,1.0
8,241708596,1.0
9,242113813,1.0


## 11.6 Define consistency validation flags

We define logical consistency checks for enriched records.

In [151]:
# [11.6] Define consistency validation function

def derive_consistency_flag(
    criminal_sophistication_score,
    behavioral_threat_score,
    opportunity_context_score
):
    # Start consistent
    consistency_flag = 1

    # Example rule:
    # impossible negative scores
    if (
        criminal_sophistication_score < 0
        or behavioral_threat_score < 0
        or opportunity_context_score < 0
    ):
        consistency_flag = 0

    return consistency_flag

## 11.7 Apply consistency validation

We apply basic logical consistency checks to enriched records.

In [152]:
# [11.7] Apply consistency validation

df["consistency_flag"] = df.apply(
    lambda row: derive_consistency_flag(
        row["criminal_sophistication_score"],
        row["behavioral_threat_score"],
        row["opportunity_context_score"]
    ),
    axis=1
)

df[
    [
        "case_id",
        "criminal_sophistication_score",
        "behavioral_threat_score",
        "opportunity_context_score",
        "consistency_flag"
    ]
].head(10)

,case_id,criminal_sophistication_score,behavioral_threat_score,opportunity_context_score,consistency_flag
0,211507896,1,2,1,1
1,201516622,4,20,3,1
2,240913563,1,2,0,1
3,210704711,0,0,0,1
4,201418201,3,9,1,1
5,240412063,1,2,0,1
6,240317069,2,4,0,1
7,201115217,3,16,0,1
8,241708596,1,2,0,1
9,242113813,1,2,0,1


## 11.8 Define enrichment confidence scoring

We define a confidence score for enrichment outputs.

In [153]:
# [11.8] Define enrichment confidence function

def calculate_enrichment_confidence(
    consistency_flag,
    completeness_score,
    source_reliability_score=0.80
):
    enrichment_confidence = (
        (
            consistency_flag
            + completeness_score
            + source_reliability_score
        )
        / 3
    )

    if enrichment_confidence >= 0.85:
        confidence_code = "high"

    elif enrichment_confidence >= 0.65:
        confidence_code = "medium"

    else:
        confidence_code = "low"

    return (
        enrichment_confidence,
        confidence_code
    )

## 11.9 Apply enrichment confidence scoring

We calculate enrichment confidence for each record using completeness, consistency, and source reliability.

In [154]:
# [11.9] Apply enrichment confidence scoring

df[
    [
        "enrichment_confidence_score",
        "enrichment_confidence_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        calculate_enrichment_confidence(
            row["consistency_flag"],
            row["completeness_score"]
        )
    ),
    axis=1
)

df[
    [
        "case_id",
        "completeness_score",
        "consistency_flag",
        "enrichment_confidence_score",
        "enrichment_confidence_code"
    ]
].head(10)

,case_id,completeness_score,consistency_flag,enrichment_confidence_score,enrichment_confidence_code
0,211507896,1.0,1,0.933333,high
1,201516622,1.0,1,0.933333,high
2,240913563,1.0,1,0.933333,high
3,210704711,1.0,1,0.933333,high
4,201418201,1.0,1,0.933333,high
5,240412063,1.0,1,0.933333,high
6,240317069,1.0,1,0.933333,high
7,201115217,1.0,1,0.933333,high
8,241708596,1.0,1,0.933333,high
9,242113813,1.0,1,0.933333,high


## 11.10 Define record anomaly flag

We define a flag for potentially anomalous or unusual records.

In [155]:
# [11.10] Define anomaly flag function

def derive_record_anomaly_flag(
    behavioral_threat_code,
    behavior_signature_rarity_code,
    enrichment_confidence_code
):
    anomaly_flag = 0

    # Rare + high threat pattern
    if (
        behavioral_threat_code == "high"
        and behavior_signature_rarity_code == "uncommon"
    ):
        anomaly_flag = 1

    # Low confidence records may also warrant review
    if enrichment_confidence_code == "low":
        anomaly_flag = 1

    return anomaly_flag

## 11.11 Apply record anomaly flag

We apply the anomaly flag to identify records that may require additional review.

In [156]:
# [11.11] Apply record anomaly flag

df["record_anomaly_flag"] = df.apply(
    lambda row: derive_record_anomaly_flag(
        row["behavioral_threat_code"],
        row["behavior_signature_rarity_code"],
        row["enrichment_confidence_code"]
    ),
    axis=1
)

df[
    [
        "case_id",
        "behavioral_threat_code",
        "behavior_signature_rarity_code",
        "enrichment_confidence_code",
        "record_anomaly_flag"
    ]
].head(12)

,case_id,behavioral_threat_code,behavior_signature_rarity_code,enrichment_confidence_code,record_anomaly_flag
0,211507896,low,common,high,0
1,201516622,high,common,high,0
2,240913563,low,common,high,0
3,210704711,low,common,high,0
4,201418201,high,common,high,0
5,240412063,low,common,high,0
6,240317069,low,common,high,0
7,201115217,high,common,high,0
8,241708596,low,common,high,0
9,242113813,low,common,high,0


## 11.12 Define analyst review priority

We define a review prioritization flag for analyst attention.

In [157]:
# [11.12] Define analyst review priority

def derive_review_priority(
    record_anomaly_flag,
    behavioral_threat_code
):
    if (
        record_anomaly_flag == 1
        and behavioral_threat_code == "high"
    ):
        return "priority_review"

    if record_anomaly_flag == 1:
        return "review"

    return "normal"

## 11.13 Apply analyst review priority

We apply review priority rules to support analyst-oriented triage.

In [158]:
# [11.13] Apply analyst review priority

df["analyst_review_priority_code"] = df.apply(
    lambda row: derive_review_priority(
        row["record_anomaly_flag"],
        row["behavioral_threat_code"]
    ),
    axis=1
)

df[
    [
        "case_id",
        "record_anomaly_flag",
        "behavioral_threat_code",
        "analyst_review_priority_code"
    ]
].head(12)

,case_id,record_anomaly_flag,behavioral_threat_code,analyst_review_priority_code
0,211507896,0,low,normal
1,201516622,0,high,normal
2,240913563,0,low,normal
3,210704711,0,low,normal
4,201418201,0,high,normal
5,240412063,0,low,normal
6,240317069,0,low,normal
7,201115217,0,high,normal
8,241708596,0,low,normal
9,242113813,0,low,normal


## 11. Quality and Validation Milestone Summary

In this section we built a data quality and validation layer including:

### Quality controls
- completeness scoring
- logical consistency validation
- enrichment confidence scoring

### Data intelligence flags
- anomalous record flag
- analyst review priority

### Quality outputs support
- filtering low-quality records
- confidence-aware modeling
- anomaly surfacing
- analyst triage support

CRIMEX now includes six major pillars:

1. Temporal enrichment
2. Geo enrichment
3. Behavioral intelligence enrichment
4. Contextual enrichment
5. Actor intelligence architecture
6. Data quality and validation intelligence

## 12. Graph and Relationship Intelligence Layer

We begin defining graph-oriented intelligence features to represent relationships among actors, incidents, locations, and behavioral patterns.

## 12.1 Define graph entity types

We define the core entity types for CRIMEX relationship intelligence.

In [159]:
# [12.1] Define graph entity types

CRIMEX_GRAPH_ENTITY_TYPES = [
    "actor",
    "incident",
    "location",
    "behavior_signature",
    "crime_type"
]

CRIMEX_GRAPH_ENTITY_TYPES

['actor', 'incident', 'location', 'behavior_signature', 'crime_type']

## 12.2 Define graph relationship types

We define core relationship types among graph entities.

In [160]:
# [12.2] Define graph relationship types

CRIMEX_GRAPH_RELATIONSHIP_TYPES = [
    "ACTOR_COMMITTED_INCIDENT",
    "INCIDENT_OCCURRED_AT_LOCATION",
    "INCIDENT_HAS_BEHAVIOR_SIGNATURE",
    "INCIDENT_HAS_CRIME_TYPE",
    "ACTOR_LINKED_TO_ACTOR"
]

CRIMEX_GRAPH_RELATIONSHIP_TYPES

['ACTOR_COMMITTED_INCIDENT',
 'INCIDENT_OCCURRED_AT_LOCATION',
 'INCIDENT_HAS_BEHAVIOR_SIGNATURE',
 'INCIDENT_HAS_CRIME_TYPE',
 'ACTOR_LINKED_TO_ACTOR']

## 12.3 Create graph node keys

We create stable node keys that can later be used for Neo4j or graph exports.

In [161]:
# [12.3] Create graph node keys

# Incident node key
df["incident_node_key"] = "incident|" + df["case_id"].astype(str)

# Location node key based on final coordinates
df["location_node_key"] = (
    "location|"
    + df["latitude_final"].round(5).astype(str)
    + "|"
    + df["longitude_final"].round(5).astype(str)
)

# Behavior signature node key
df["behavior_signature_node_key"] = (
    "behavior_signature|"
    + df["behavior_signature_key"].astype(str)
)

# Crime type node key
df["crime_type_node_key"] = (
    "crime_type|"
    + df["crime_ontology_code"].astype(str)
)

df[
    [
        "incident_node_key",
        "location_node_key",
        "behavior_signature_node_key",
        "crime_type_node_key"
    ]
].head(10)

,incident_node_key,location_node_key,behavior_signature_node_key,crime_type_node_key
0,incident|211507896,location|34.2124|-118.4092,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity
1,incident|201516622,location|34.1993|-118.4203,behavior_signature|violent|public_space|violen...,crime_type|violent
2,incident|240913563,location|34.1847|-118.4509,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity
3,incident|210704711,location|34.0339|-118.3747,behavior_signature|property|vehicle|opportunis...,crime_type|property
4,incident|201418201,location|33.9813|-118.435,behavior_signature|property|vehicle|organized_...,crime_type|property
5,incident|240412063,location|34.083|-118.1678,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity
6,incident|240317069,location|34.01|-118.29,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity
7,incident|201115217,location|34.1107|-118.2589,behavior_signature|child_related|child|vulnera...,crime_type|child_related
8,incident|241708596,location|34.2763|-118.521,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity
9,incident|242113813,location|34.1493|-118.5886,behavior_signature|fraud_identity|identity|str...,crime_type|fraud_identity


## 12.4 Create behavior signature connectivity proxy

We derive a proxy for connectivity of recurring behavioral signatures.

In [162]:
# [12.4] Create behavior signature connectivity proxy

behavior_counts = (
    df["behavior_signature_key"]
    .value_counts()
)

df[
    "behavior_signature_connectivity_degree"
] = df[
    "behavior_signature_key"
].map(
    behavior_counts
)

def map_connectivity_code(degree):

    if degree >= 1000:
        return "high"

    if degree >= 200:
        return "medium"

    return "low"


df[
    "behavior_signature_connectivity_code"
] = df[
    "behavior_signature_connectivity_degree"
].apply(
    map_connectivity_code
)

df[
    [
        "behavior_signature_key",
        "behavior_signature_connectivity_degree",
        "behavior_signature_connectivity_code"
    ]
].head(10)

,behavior_signature_key,behavior_signature_connectivity_degree,behavior_signature_connectivity_code
0,fraud_identity|identity|structured_fraud|simple|0,19587,high
1,violent|public_space|violent_high_risk|complex|1,12921,high
2,fraud_identity|identity|structured_fraud|simple|0,19587,high
3,property|vehicle|opportunistic_property|simple|0,16802,high
4,property|vehicle|organized_complex|complex|0,46707,high
5,fraud_identity|identity|structured_fraud|simple|0,19587,high
6,fraud_identity|identity|structured_fraud|moder...,34165,high
7,child_related|child|vulnerable_targeted|complex|0,2549,high
8,fraud_identity|identity|structured_fraud|simple|0,19587,high
9,fraud_identity|identity|structured_fraud|simple|0,19587,high


## 12.5 Define behavioral centrality risk

We derive a graph-inspired behavioral centrality risk proxy.

In [163]:
# [12.5] Define behavioral centrality risk function

def derive_behavior_centrality_risk(
    connectivity_code,
    behavioral_threat_code,
    behavior_signature_rarity_code
):
    risk_score = 0

    # Rare pattern contribution
    if behavior_signature_rarity_code == "uncommon":
        risk_score += 1

    # Low connectivity unusual pattern
    if connectivity_code == "low":
        risk_score += 1

    # High threat contribution
    if behavioral_threat_code == "high":
        risk_score += 1

    if risk_score <= 1:
        risk_code = "low"

    elif risk_score == 2:
        risk_code = "medium"

    else:
        risk_code = "high"

    return (
        risk_score,
        risk_code
    )

## 12.6 Apply behavioral centrality risk

We apply a graph-inspired behavioral centrality risk proxy.

In [164]:
# [12.6] Apply behavioral centrality risk

df[
    [
        "behavior_centrality_risk_score",
        "behavior_centrality_risk_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_behavior_centrality_risk(
            row["behavior_signature_connectivity_code"],
            row["behavioral_threat_code"],
            row["behavior_signature_rarity_code"]
        )
    ),
    axis=1
)

df[
    [
        "behavior_signature_connectivity_code",
        "behavior_signature_rarity_code",
        "behavioral_threat_code",
        "behavior_centrality_risk_score",
        "behavior_centrality_risk_code"
    ]
].head(12)

,behavior_signature_connectivity_code,behavior_signature_rarity_code,behavioral_threat_code,behavior_centrality_risk_score,behavior_centrality_risk_code
0,high,common,low,0,low
1,high,common,high,1,low
2,high,common,low,0,low
3,high,common,low,0,low
4,high,common,high,1,low
5,high,common,low,0,low
6,high,common,low,0,low
7,high,common,high,1,low
8,high,common,low,0,low
9,high,common,low,0,low


## 12.7 Define co-offending risk proxy

We derive a proxy for co-offending or organized group-linked behavior.

In [165]:
# [12.7] Define co-offending risk proxy function

def derive_cooffending_risk(
    behavior_risk_profile_code,
    offender_persistence_proxy_code
):
    risk_score = 0

    if behavior_risk_profile_code == "organized_complex":
        risk_score += 1

    if offender_persistence_proxy_code == "high":
        risk_score += 1

    if risk_score == 0:
        risk_code = "low"

    elif risk_score == 1:
        risk_code = "medium"

    else:
        risk_code = "high"

    return (
        risk_score,
        risk_code
    )

## 12.8 Apply co-offending risk proxy

We apply a proxy for co-offending and organized group-linked behavior.

In [166]:
# [12.8] Apply co-offending risk proxy

df[
    [
        "cooffending_risk_score",
        "cooffending_risk_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_cooffending_risk(
            row["behavior_risk_profile_code"],
            row["offender_persistence_proxy_code"]
        )
    ),
    axis=1
)

df[
    [
        "behavior_risk_profile_code",
        "offender_persistence_proxy_code",
        "cooffending_risk_score",
        "cooffending_risk_code"
    ]
].head(12)

,behavior_risk_profile_code,offender_persistence_proxy_code,cooffending_risk_score,cooffending_risk_code
0,structured_fraud,low,0,low
1,violent_high_risk,medium,0,low
2,structured_fraud,low,0,low
3,opportunistic_property,low,0,low
4,organized_complex,high,2,high
5,structured_fraud,low,0,low
6,structured_fraud,low,0,low
7,vulnerable_targeted,medium,0,low
8,structured_fraud,low,0,low
9,structured_fraud,low,0,low


## 12.9 Create location connectivity proxy

We derive a connectivity proxy for repeat incident concentration at locations.

In [167]:
# [12.9] Create location connectivity proxy

location_counts = (
    df["location_node_key"]
    .value_counts()
)

df[
    "location_connectivity_degree"
] = df[
    "location_node_key"
].map(
    location_counts
)

def map_location_connectivity_code(degree):

    if degree >= 5:
        return "high"

    if degree >= 2:
        return "medium"

    return "low"


df[
    "location_connectivity_code"
] = df[
    "location_connectivity_degree"
].apply(
    map_location_connectivity_code
)

df[
    [
        "location_node_key",
        "location_connectivity_degree",
        "location_connectivity_code"
    ]
].head(10)

,location_node_key,location_connectivity_degree,location_connectivity_code
0,location|34.2124|-118.4092,6.0,high
1,location|34.1993|-118.4203,15.0,high
2,location|34.1847|-118.4509,44.0,high
3,location|34.0339|-118.3747,27.0,high
4,location|33.9813|-118.435,25.0,high
5,location|34.083|-118.1678,15.0,high
6,location|34.01|-118.29,32.0,high
7,location|34.1107|-118.2589,3.0,medium
8,location|34.2763|-118.521,1.0,low
9,location|34.1493|-118.5886,4.0,medium


## 12.10 Define network exposure risk proxy

We derive a composite graph-oriented network exposure risk proxy.

In [168]:
# [12.10] Define network exposure risk proxy function

def derive_network_exposure_risk(
    behavior_centrality_risk_code,
    cooffending_risk_code,
    location_connectivity_code
):
    risk_score = 0

    if behavior_centrality_risk_code == "high":
        risk_score += 1

    if cooffending_risk_code == "high":
        risk_score += 1

    # medium OR high hotspot connectivity
    if location_connectivity_code in [
        "medium",
        "high"
    ]:
        risk_score += 1

    if risk_score == 0:
        risk_code = "low"

    elif risk_score == 1:
        risk_code = "medium"

    else:
        risk_code = "high"

    return (
        risk_score,
        risk_code
    )

## 12.11 Apply network exposure risk proxy

We apply a composite graph-oriented network exposure risk proxy.

In [169]:
# [12.11] Apply network exposure risk proxy

df[
    [
        "network_exposure_risk_score",
        "network_exposure_risk_code"
    ]
] = df.apply(
    lambda row: pd.Series(
        derive_network_exposure_risk(
            row["behavior_centrality_risk_code"],
            row["cooffending_risk_code"],
            row["location_connectivity_code"]
        )
    ),
    axis=1
)

df[
    [
        "behavior_centrality_risk_code",
        "cooffending_risk_code",
        "location_connectivity_code",
        "network_exposure_risk_score",
        "network_exposure_risk_code"
    ]
].head(12)

,behavior_centrality_risk_code,cooffending_risk_code,location_connectivity_code,network_exposure_risk_score,network_exposure_risk_code
0,low,low,high,1,medium
1,low,low,high,1,medium
2,low,low,high,1,medium
3,low,low,high,1,medium
4,low,high,high,2,high
5,low,low,high,1,medium
6,low,low,high,1,medium
7,low,low,medium,1,medium
8,low,low,low,0,low
9,low,low,medium,1,medium


## 12. Graph Intelligence Milestone Summary

In this section we built graph-oriented intelligence features including:

### Graph foundations
- graph entity types
- graph relationship types
- stable node keys for graph export

### Graph intelligence features
- behavior signature connectivity proxy
- behavioral centrality risk proxy
- co-offending risk proxy
- location connectivity / hotspot proxy
- composite network exposure risk proxy

These features support:
- graph-ready exports for Neo4j
- behavioral network analysis
- hotspot connectivity analysis
- organized behavior signals
- graph anomaly and exposure scoring

CRIMEX now includes seven major intelligence pillars:

1. Temporal enrichment  
2. Geo enrichment  
3. Behavioral intelligence enrichment  
4. Contextual enrichment  
5. Actor intelligence architecture  
6. Data quality and validation intelligence  
7. Graph and relationship intelligence

## 13. Explainability and Risk Reasoning Layer

We create explanation fields that describe why a case receives elevated behavioral, contextual, or network risk scores.

## 13.1 Define risk explanation function

We define a rule-based explanation function for elevated risk indicators.

In [170]:
# [13.1] Define risk explanation function

def build_risk_explanation(row):

    reasons = []

    if row["behavioral_threat_code"] == "high":
        reasons.append("high behavioral threat")

    if row["escalation_potential_code"] == "high":
        reasons.append("high escalation potential")

    if row["routine_activity_risk_code"] == "high":
        reasons.append("high routine activity risk")

    # FIX:
    if row["network_exposure_risk_code"] in [
        "medium",
        "high"
    ]:
        reasons.append("elevated network exposure risk")

    if row["behavioral_anomaly_flag"] == 1:
        reasons.append("behavioral anomaly detected")

    if len(reasons) == 0:
        return "no elevated risk indicators detected"

    return " | ".join(reasons)

## 13.2 Apply risk explanation field

We generate an interpretable explanation field for each record.

In [171]:
# [13.2] Apply risk explanation field

df["risk_explanation_text"] = df.apply(
    build_risk_explanation,
    axis=1
)

df[
    [
        "crime_description",
        "behavioral_threat_code",
        "network_exposure_risk_code",
        "risk_explanation_text"
    ]
].head(12)

,crime_description,behavioral_threat_code,network_exposure_risk_code,risk_explanation_text
0,THEFT OF IDENTITY,low,medium,elevated network exposure risk
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",high,medium,high behavioral threat | high escalation poten...
2,THEFT OF IDENTITY,low,medium,elevated network exposure risk
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,low,medium,elevated network exposure risk
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),high,high,high behavioral threat | elevated network expo...
5,THEFT OF IDENTITY,low,medium,elevated network exposure risk
6,THEFT OF IDENTITY,low,medium,elevated network exposure risk
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,high,medium,high behavioral threat | elevated network expo...
8,THEFT OF IDENTITY,low,low,no elevated risk indicators detected
9,THEFT OF IDENTITY,low,medium,elevated network exposure risk


## 13.3 Create explanation signature key

We create a compact explanation signature key summarizing major risk drivers.

In [172]:
# [13.3] Create explanation signature key

def build_explanation_signature(row):

    parts = []

    if row["behavioral_threat_code"] == "high":
        parts.append("THREAT")

    if row["routine_activity_risk_code"] == "high":
        parts.append("ROUTINE")

    if row["network_exposure_risk_code"] in [
        "medium",
        "high"
    ]:
        parts.append("NETWORK")

    if row["behavioral_anomaly_flag"] == 1:
        parts.append("ANOMALY")

    if len(parts) == 0:
        return "BASELINE"

    return "|".join(parts)

## 13.4 Apply explanation signature key

We generate a compact machine-readable explanation signature for each record.

In [173]:
# [13.4] Apply explanation signature key

df["risk_explanation_signature_code"] = df.apply(
    build_explanation_signature,
    axis=1
)

df[
    [
        "crime_description",
        "risk_explanation_text",
        "risk_explanation_signature_code"
    ]
].head(12)

,crime_description,risk_explanation_text,risk_explanation_signature_code
0,THEFT OF IDENTITY,elevated network exposure risk,NETWORK
1,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",high behavioral threat | high escalation poten...,THREAT|ROUTINE|NETWORK|ANOMALY
2,THEFT OF IDENTITY,elevated network exposure risk,NETWORK
3,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,elevated network exposure risk,NETWORK
4,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),high behavioral threat | elevated network expo...,THREAT|NETWORK
5,THEFT OF IDENTITY,elevated network exposure risk,NETWORK
6,THEFT OF IDENTITY,elevated network exposure risk,NETWORK
7,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,high behavioral threat | elevated network expo...,THREAT|NETWORK
8,THEFT OF IDENTITY,no elevated risk indicators detected,BASELINE
9,THEFT OF IDENTITY,elevated network exposure risk,NETWORK


## 13. Explainability and Risk Reasoning Milestone Summary

In this section we built an explainability layer including:

### Explainability features
- human-readable risk explanation text
- machine-readable explanation signature codes

### Reasoning outputs support
- interpretable risk rationale
- explanation clustering
- analyst understanding of why a case is elevated
- validation of consistency between risk signals

This layer helps bridge:
- feature engineering
- intelligence reasoning
- analyst interpretation
- future explainable modeling

## 14. Global Dataset Expansion Layer

We define structures supporting CRIMEX as a large-scale multilingual worldwide criminal behavior dataset.

## 14.1 Define global enrichment dimensions

We define global enrichment dimensions for worldwide crime behavior data integration.

In [174]:
# [14.1] Define global enrichment dimensions

CRIMEX_GLOBAL_DIMENSIONS = [

    "multilingual_crime_ontology",

    "country_specific_legal_context",

    "cross_country_behavior_patterns",

    "region_specific_crime_modus_operandi",

    "country_specific_holiday_context",

    "country_specific_environment_context"

]

CRIMEX_GLOBAL_DIMENSIONS

['multilingual_crime_ontology',
 'country_specific_legal_context',
 'cross_country_behavior_patterns',
 'region_specific_crime_modus_operandi',
 'country_specific_holiday_context',
 'country_specific_environment_context']

## 14.2 Define crime ontology hierarchy levels

We define hierarchical ontology levels for global crime taxonomy harmonization.

In [175]:
# [14.2] Define crime ontology hierarchy

CRIMEX_CRIME_ONTOLOGY_LEVELS = {

    "level_1_macro_categories": [
        "violent",
        "property",
        "fraud_identity",
        "cyber",
        "drug",
        "child_related"
    ],

    "level_2_behavior_classes": [
        "organized_complex",
        "opportunistic_property",
        "structured_fraud",
        "violent_high_risk"
    ],

    "level_3_modus_operandi_patterns":
        "region or offense specific MO patterns"
}

CRIMEX_CRIME_ONTOLOGY_LEVELS

{'level_1_macro_categories': ['violent',
  'property',
  'fraud_identity',
  'cyber',
  'drug',
  'child_related'],
 'level_2_behavior_classes': ['organized_complex',
  'opportunistic_property',
  'structured_fraud',
  'violent_high_risk'],
 'level_3_modus_operandi_patterns': 'region or offense specific MO patterns'}

## 14.3 Define source coverage metadata

We define metadata describing global source coverage of the dataset.

In [176]:
# [14.3] Define source coverage metadata

CRIMEX_SOURCE_COVERAGE_FIELDS = [
    "source_country_code",
    "source_region_code",
    "source_jurisdiction_type",
    "crime_reporting_source_type",
    "source_coverage_scope_code",
    "source_record_count"
]

CRIMEX_SOURCE_COVERAGE_FIELDS

['source_country_code',
 'source_region_code',
 'source_jurisdiction_type',
 'crime_reporting_source_type',
 'source_coverage_scope_code',
 'source_record_count']

## 14.4 Define legal harmonization layer

We define fields supporting harmonization between country-specific legal labels and CRIMEX ontology.

In [177]:
# [14.4] Define legal harmonization fields

CRIMEX_LEGAL_HARMONIZATION_FIELDS = [
    "local_offense_label",
    "local_legal_code",
    "crime_ontology_code",
    "harmonization_confidence_score",
    "legal_mapping_source_code"
]

CRIMEX_LEGAL_HARMONIZATION_FIELDS

['local_offense_label',
 'local_legal_code',
 'crime_ontology_code',
 'harmonization_confidence_score',
 'legal_mapping_source_code']

## 14.5 Define regional MO library structure

We define a structure for region-specific modus operandi enrichment.

In [178]:
# [14.5] Define regional MO library fields

CRIMEX_REGIONAL_MO_FIELDS = [
    "region_mo_pattern_code",
    "region_mo_pattern_desc",
    "region_behavior_signature_variant",
    "region_mo_risk_score"
]

CRIMEX_REGIONAL_MO_FIELDS

['region_mo_pattern_code',
 'region_mo_pattern_desc',
 'region_behavior_signature_variant',
 'region_mo_risk_score']

## 14.6 Define multilingual offense normalization lexicon

We define a multilingual lexicon layer for offense normalization.

In [179]:
# [14.6] Define multilingual offense lexicon fields

CRIMEX_OFFENSE_LEXICON_FIELDS = [
    "offense_surface_form",
    "offense_language_code",
    "offense_synonym_group_id",
    "crime_ontology_code",
    "normalization_confidence_score"
]

CRIMEX_OFFENSE_LEXICON_FIELDS

['offense_surface_form',
 'offense_language_code',
 'offense_synonym_group_id',
 'crime_ontology_code',
 'normalization_confidence_score']

## 14.7 Define representation diagnostics layer

We define fields for analyzing dataset representation balance and coverage bias.

In [180]:
# [14.7] Define representation diagnostics fields

CRIMEX_REPRESENTATION_DIAGNOSTICS = [
    "country_coverage_density",
    "crime_family_coverage_density",
    "language_coverage_density",
    "source_diversity_score",
    "representation_bias_flag"
]

CRIMEX_REPRESENTATION_DIAGNOSTICS

['country_coverage_density',
 'crime_family_coverage_density',
 'language_coverage_density',
 'source_diversity_score',
 'representation_bias_flag']

## 14.8 Define downstream task support layer

We define task support metadata for downstream uses of CRIMEX.

In [181]:
# [14.8] Define downstream task support metadata

CRIMEX_TASK_SUPPORT = [
    "crime_risk_prediction_support",
    "behavior_pattern_analysis_support",
    "anomaly_detection_support",
    "criminal_network_analysis_support",
    "link_prediction_support",
    "cross_country_behavior_comparison_support"
]

CRIMEX_TASK_SUPPORT

['crime_risk_prediction_support',
 'behavior_pattern_analysis_support',
 'anomaly_detection_support',
 'criminal_network_analysis_support',
 'link_prediction_support',
 'cross_country_behavior_comparison_support']

## 14. Global Dataset Expansion Milestone Summary

In this section we expanded CRIMEX toward a worldwide criminal behavior benchmark through:

### Global dataset architecture
- global enrichment dimensions
- crime ontology hierarchy
- source coverage metadata
- legal harmonization layer
- regional modus operandi library
- multilingual offense normalization lexicon

### Dataset diagnostics
- representation diagnostics
- coverage and bias awareness

### Multi-task benchmark support
- risk prediction support
- anomaly detection support
- network analysis support
- link prediction support
- cross-country behavior comparison support

CRIMEX now extends beyond a dataset into:

- criminal behavior ontology
- multilingual intelligence benchmark
- graph-ready knowledge resource
- global criminal behavior dataset framework

## 15. Final CRIMEX Dataset Deliverables

We define the planned deliverable artifacts produced by the CRIMEX pipeline.

In [182]:
# [15.1] Define CRIMEX deliverables

CRIMEX_DELIVERABLES = [
    "crimex_core_dataset",
    "crimex_graph_nodes",
    "crimex_graph_edges",
    "crimex_crime_ontology",
    "crimex_metadata_catalog",
    "crimex_pipeline_configuration"
]

CRIMEX_DELIVERABLES

['crimex_core_dataset',
 'crimex_graph_nodes',
 'crimex_graph_edges',
 'crimex_crime_ontology',
 'crimex_metadata_catalog',
 'crimex_pipeline_configuration']

## 15.2 Define final CRIMEX core dataset sections

We define the main sections included in the final CRIMEX core dataset.

In [183]:
# [15.2] Define final CRIMEX core dataset sections

CRIMEX_CORE_DATASET_SECTIONS = [
    "source_provenance_features",
    "incident_core_features",
    "temporal_features",
    "geo_features",
    "behavior_features",
    "context_features",
    "actor_features",
    "quality_validation_features",
    "graph_intelligence_features",
    "explainability_features"
]

CRIMEX_CORE_DATASET_SECTIONS

['source_provenance_features',
 'incident_core_features',
 'temporal_features',
 'geo_features',
 'behavior_features',
 'context_features',
 'actor_features',
 'quality_validation_features',
 'graph_intelligence_features',
 'explainability_features']

## 15.3 Define export package artifacts

We define the final exported artifacts produced by CRIMEX.

In [184]:
# [15.3] Define export package artifacts

CRIMEX_EXPORT_PACKAGE = {
    "tabular_dataset":
        "crimex_core_dataset.parquet",

    "graph_nodes":
        "crimex_graph_nodes.csv",

    "graph_edges":
        "crimex_graph_edges.csv",

    "ontology_resource":
        "crimex_crime_ontology.json",

    "metadata_catalog":
        "crimex_metadata_catalog.json",

    "pipeline_config":
        "crimex_pipeline_config.yaml"
}

CRIMEX_EXPORT_PACKAGE

{'tabular_dataset': 'crimex_core_dataset.parquet',
 'graph_nodes': 'crimex_graph_nodes.csv',
 'graph_edges': 'crimex_graph_edges.csv',
 'ontology_resource': 'crimex_crime_ontology.json',
 'metadata_catalog': 'crimex_metadata_catalog.json',
 'pipeline_config': 'crimex_pipeline_config.yaml'}

## 15.4 Define dataset card metadata

We define metadata fields describing the CRIMEX dataset release.

In [185]:
# [15.4] Define dataset card metadata fields

CRIMEX_DATASET_CARD_FIELDS = [
    "dataset_description",
    "countries_covered",
    "languages_covered",
    "crime_families_covered",
    "number_of_records",
    "number_of_features",
    "source_types",
    "supported_tasks",
    "license"
]

CRIMEX_DATASET_CARD_FIELDS

['dataset_description',
 'countries_covered',
 'languages_covered',
 'crime_families_covered',
 'number_of_records',
 'number_of_features',
 'source_types',
 'supported_tasks',
 'license']

## 15. Final CRIMEX Dataset Design Summary

CRIMEX final deliverables include:

### Core artifacts
- CRIMEX enriched core dataset
- graph nodes and graph edges exports
- crime ontology resource
- metadata catalog
- pipeline configuration package

### Final core dataset sections
- source provenance
- incident core
- temporal
- geo
- behavioral
- contextual
- actor intelligence
- quality validation
- graph intelligence
- explainability

### Final architecture capabilities
CRIMEX supports:

- multilingual global crime data integration
- criminal behavior intelligence enrichment
- graph-ready criminal relationship data
- explainable intelligence signals
- scalable reusable enrichment pipelines
- future machine learning and criminal behavior prediction tasks

CRIMEX is designed as:

- dataset
- ontology resource
- graph intelligence resource
- reusable criminal intelligence data pipeline

In [186]:
df.shape

(1004894, 128)

In [187]:
df.head(3)

,case_id,date_reported,date_occured,time_occured,area_code,area_name,report_district,crime_part,crime_code,crime_description,...,behavior_centrality_risk_score,behavior_centrality_risk_code,cooffending_risk_score,cooffending_risk_code,location_connectivity_degree,location_connectivity_code,network_exposure_risk_score,network_exposure_risk_code,risk_explanation_text,risk_explanation_signature_code
0,211507896,2021-04-11,2020-11-07,08:45:00,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,0,low,0,low,6.0,high,1,medium,elevated network exposure risk,NETWORK
1,201516622,2020-10-21,2020-10-18,18:45:00,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,1,low,0,low,15.0,high,1,medium,high behavioral threat | high escalation poten...,THREAT|ROUTINE|NETWORK|ANOMALY
2,240913563,2024-12-10,2020-10-30,12:40:00,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,0,low,0,low,44.0,high,1,medium,elevated network exposure risk,NETWORK


## 16. Save CRIMEX Checkpoint Release

We save a versioned CRIMEX dataset checkpoint and supporting artifacts.

In [188]:
# [16.1] Save CRIMEX dataset checkpoint

from pathlib import Path

OUTPUT_DIR = Path("../data/final")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_VERSION = "v1_checkpoint"

parquet_path = OUTPUT_DIR / f"crimex_{DATASET_VERSION}_v21.parquet"
csv_path = OUTPUT_DIR / f"crimex_{DATASET_VERSION}_v21.csv"

df.to_parquet(
    parquet_path,
    index=False
)

df.to_csv(
    csv_path,
    index=False
)

print("Saved:")
print(parquet_path)
print(csv_path)

Saved:
data\final\crimex_v1_checkpoint_v21.parquet
data\final\crimex_v1_checkpoint_v21.csv


In [189]:
# [16.2] Save schema inventory

import json

schema_dict = {
    "row_count": int(df.shape[0]),
    "column_count": int(df.shape[1]),
    "columns": list(df.columns)
}

schema_path = OUTPUT_DIR / "crimex_schema_v1_v21.json"

with open(schema_path,"w",encoding="utf-8") as f:
    json.dump(
        schema_dict,
        f,
        indent=2,
        ensure_ascii=False
    )

print(schema_path)

data\final\crimex_schema_v1_v21.json


In [190]:
# [16.3] Save feature group manifest

feature_manifest = {
    "dataset_sections":
        CRIMEX_CORE_DATASET_SECTIONS
}

manifest_path = (
    OUTPUT_DIR /
    "crimex_feature_manifest_v1_v2.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_manifest,
        f,
        indent=2
    )

print(manifest_path)

data\final\crimex_feature_manifest_v1_v2.json


In [191]:
df.describe(include="all").to_csv(
    OUTPUT_DIR / "crimex_profile_summary_v2.csv"
)